# Proyecto final de Machine Learning — Plan de neveras FEMSA

**Pregunta de negocio:** ¿en cuáles **1.000 tiendas** debe FEMSA instalar sus 1.000 neveras nuevas, y cuánto ganaría cada mes?

Este cuaderno corre **de principio a fin** desde la base original (`base_refrescos_nevera.csv`) y produce
todos los archivos que alimentan el tablero y la entrega.

---

## Cómo se organiza

| Módulo | Contenido | Puntos |
|---|---|---|
| 0 | Configuración, carga y validación de la base | — |
| 1 | Análisis exploratorio | 10 |
| 2 | Preparación de datos y control de fuga | 10 |
| 3 | Modelo de ventas (3+ algoritmos, validación cruzada) | 15 |
| 4 | **Efecto causal de la nevera** (el corazón del proyecto) | 20 |
| 5 | Riesgo de cambio de proveedor | 10 |
| 6 | Decisión: las 1.000 tiendas y la ganancia | 10 |
| 7 | Exportación de resultados para el tablero | — |

## Supuestos económicos del caso

| Supuesto | Valor |
|---|---|
| Margen de FEMSA sobre ventas al público | 25 % |
| Costo de cada nevera | COP 60.000 / mes |
| Valor de evitar un cambio de proveedor | COP 500.000 |
| Costo de una acción de retención | COP 50.000 |
| Efectividad de la retención | 40 % |

$$\text{Ganancia mensual} = 0{,}25 \times \text{Ventas adicionales por la nevera} - 60.000$$

**Tienda elegible:** `nevera_femsa = 0`, excepto si tiene nevera de Postobón y menos de 12 m².

## Reproducibilidad

Semilla fija `SEED = 42` en todos los modelos y particiones. Correr *Entorno de ejecución → Ejecutar todo*
reproduce exactamente los mismos números y los mismos 1.000 id_tienda.

---

### Declaración de uso de IA

Este proyecto se desarrolló con apoyo de un asistente de IA (Claude) en las siguientes tareas:
exploración inicial de la base, redacción del código del cuaderno, elección y discusión de los
estimadores causales, y redacción de los textos explicativos. Las decisiones de modelación
(definición del tratamiento, variables excluidas por fuga, estimador final, criterio de selección de las
1.000 tiendas) fueron revisadas, verificadas y asumidas por el grupo. Todos los integrantes pueden
explicar y modificar cualquier celda de este cuaderno.

---
# Módulo 0 — Configuración y carga de datos

In [ ]:
# Dependencias. En Google Colab pandas, numpy, scikit-learn y matplotlib ya vienen instalados.
# Esta celda solo asegura una version minima de scikit-learn.
import sys, subprocess
try:
    import sklearn
    assert tuple(int(x) for x in sklearn.__version__.split('.')[:2]) >= (1, 3)
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn>=1.3'], check=False)

import numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt, sklearn, warnings, os, json, textwrap
warnings.filterwarnings('ignore')
print('pandas', pd.__version__, '| numpy', np.__version__, '| scikit-learn', sklearn.__version__)

In [ ]:
# ----------------------------- PARAMETROS DEL PROYECTO -----------------------------
SEED = 42                      # semilla fija: reproducibilidad
np.random.seed(SEED)

MARGEN               = 0.25     # margen de FEMSA sobre ventas al publico
COSTO_NEVERA         = 60_000   # COP por nevera por mes
VALOR_EVITAR_CAMBIO  = 500_000  # COP por evitar un cambio de proveedor
COSTO_RETENCION      = 50_000   # COP por accion de retencion
EFECTIVIDAD_RETENCION= 0.40     # 40 %
N_NEVERAS            = 1_000    # neveras a instalar
AREA_MIN_SI_POSTOBON = 12.0     # m2: si tiene nevera Postobon y area < 12 no es elegible

GRUPO = 'X'                     # <<< CAMBIAR por el numero de su grupo (ej. '3')

# Estetica de los graficos
plt.rcParams.update({'figure.dpi':110, 'figure.figsize':(9,4.5), 'axes.grid':True,
                     'grid.alpha':.25, 'axes.spines.top':False, 'axes.spines.right':False,
                     'font.size':10, 'axes.titlesize':11, 'axes.titleweight':'bold'})
C_SIN, C_CON, C_ML, C_R1, C_R2 = '#d1495b', '#0b7285', '#2b8a3e', '#e8973a', '#8d6ab5'

def cop(x, dec=0):
    'Formatea un numero como pesos colombianos.'
    return f'$ {x:,.{dec}f}'.replace(',', '.')

def mm(x):
    'Formatea en millones de COP.'
    return f'{x/1e6:,.1f} MM'.replace(',', '.')

RUTA_SALIDAS = 'salidas'
os.makedirs(RUTA_SALIDAS, exist_ok=True)
print('Parametros cargados. Semilla =', SEED)

### Carga de la base

La celda busca el archivo en varias rutas. En Colab hay tres opciones:

1. **Subir el archivo** (más simple): se abre el selector si no encuentra la base.
2. **Google Drive**: descomentar el bloque de `drive.mount` y ajustar `RUTA_DRIVE`.
3. **Ruta local**: si se corre fuera de Colab, dejar el CSV en `datos/`.

In [ ]:
RUTA_DRIVE = '/content/drive/MyDrive/proyecto_ml/base_refrescos_nevera.csv'  # ajustar si usan Drive

# --- Opcion 2: Google Drive (descomentar las 2 lineas) ---
# from google.colab import drive
# drive.mount('/content/drive')

CANDIDATAS = ['base_refrescos_nevera.csv', 'datos/base_refrescos_nevera.csv',
              '/content/base_refrescos_nevera.csv', RUTA_DRIVE]

ruta = next((p for p in CANDIDATAS if os.path.exists(p)), None)

if ruta is None:
    try:                                    # Opcion 1: subir el archivo en Colab
        from google.colab import files
        print('Suba base_refrescos_nevera.csv')
        sub = files.upload()
        ruta = list(sub.keys())[0]
    except ImportError:
        raise FileNotFoundError('Coloque base_refrescos_nevera.csv en el directorio de trabajo.')

df = pd.read_csv(ruta)
print(f'Base cargada desde: {ruta}')
print(f'Filas: {df.shape[0]:,}   Columnas: {df.shape[1]}')

In [ ]:
# ----------------------------- VALIDACION DE INTEGRIDAD -----------------------------
chequeos = {
    'filas = 38.000'                  : len(df) == 38_000,
    'columnas = 109'                  : df.shape[1] == 109,
    'id_tienda sin duplicados'        : df['id_tienda'].duplicated().sum() == 0,
    'sin valores faltantes'           : int(df.isna().sum().sum()) == 0,
    'objetivo ventas presente'        : 'ventas_femsa_cop_mes' in df.columns,
    'objetivo cambio presente'        : 'cambio_proveedor_principal_6m' in df.columns,
    'tratamiento nevera_femsa binario': set(df['nevera_femsa'].unique()) <= {0, 1},
}
for k, v in chequeos.items():
    print(('  OK  ' if v else ' FALLA') + f' | {k}')
assert all(chequeos.values()), 'La base no cumple la estructura esperada.'
print('\nBase validada: 0 faltantes, 0 duplicados.')

---
# Módulo 1 — Análisis exploratorio

**Objetivo:** entender las tiendas, las ventas y las neveras, y mostrar en qué se diferencian las tiendas
con nevera y sin nevera. Esa diferencia es la que justifica todo el módulo 4.

Buscamos cinco cosas concretas para la decisión:

1. ¿Cómo se distribuyen las ventas de FEMSA? (define cómo modelamos)
2. ¿Cuántas tiendas son elegibles y cómo se ven?
3. ¿En qué se diferencian las tiendas con y sin nevera? (sesgo de selección)
4. ¿Qué variables parecen amplificar el efecto de una nevera? (heterogeneidad)
5. ¿Cuánto riesgo de cambio de proveedor hay?

In [ ]:
# ----------------------------- 1.1 Panorama general -----------------------------
grupos = {
 'Identificación y geografía': ['id_tienda','region','departamento','ciudad','latitud','longitud'],
 'Entorno'    : ['zona','estrato','temperatura_media_c','poblacion_500m','n_tiendas_competencia_500m',
                 'dist_hard_discount_km','cerca_colegio','cerca_parada_transporte',
                 'cerca_universidad_oficinas','dist_cedi_km'],
 'Tienda'     : ['tipo_negocio','area_m2','antiguedad_anios','dias_abre_semana','horas_abre_dia',
                 'num_empleados','tiene_datafono','acepta_pago_digital','ofrece_domicilios',
                 'tiene_mesas_consumo','vende_licor','n_referencias_total','nevera_propia_tienda',
                 'nevera_otras_marcas'],
 'Tráfico'    : ['clientes_prom_dia'],
 'Gestión comercial': ['visitas_mes_femsa','visitas_mes_postobon','pct_quiebre_femsa','pct_quiebre_postobon',
                 'credito_femsa','credito_postobon','material_pop_femsa','material_pop_postobon',
                 'promos_activas_femsa','promos_activas_postobon','indice_precio_femsa','indice_precio_postobon'],
 'Neveras'    : ['nevera_femsa','nevera_postobon','num_neveras_femsa','num_neveras_postobon',
                 'presencia_neveras','puertas_nevera_femsa','puertas_nevera_postobon',
                 'antiguedad_nevera_femsa_anios','antiguedad_nevera_postobon_anios',
                 'nevera_femsa_operativa','nevera_postobon_operativa'],
}
grupos['Resultados'] = [c for c in df.columns
                        if c not in sum(grupos.values(), []) and c != 'cambio_proveedor_principal_6m']
grupos['Objetivo'] = ['cambio_proveedor_principal_6m']

print('Columnas por grupo del diccionario:')
for g, cols in grupos.items():
    print(f'  {g:28s} {len(cols):>3} columnas')
print(f'  {"TOTAL":28s} {sum(len(c) for c in grupos.values()):>3}')

print('\nCobertura geográfica y de formato:')
for c in ['region','departamento','ciudad','tipo_negocio','zona']:
    print(f'  {c:14s} {df[c].nunique():>3} valores   ej.: {", ".join(map(str, df[c].unique()[:3]))}')

In [ ]:
# ----------------------------- 1.2 Distribucion de las ventas de FEMSA -----------------------------
v = df['ventas_femsa_cop_mes']
print('Ventas FEMSA por tienda-mes (COP):')
print(f'  media     {cop(v.mean())}')
print(f'  mediana   {cop(v.median())}')
print(f'  P5 - P95  {cop(v.quantile(.05))}  a  {cop(v.quantile(.95))}')
print(f'  maximo    {cop(v.max())}')
print(f'  asimetria en niveles  {v.skew():.2f}')
print(f'  asimetria en log      {np.log(v).skew():.2f}   <-- casi simetrica')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(v/1e6, bins=80, color=C_CON, alpha=.85)
ax[0].set(title='Ventas FEMSA mensuales (millones COP)', xlabel='MM COP / mes', ylabel='tiendas')
ax[0].set_xlim(0, 10)
ax[1].hist(np.log(v), bins=80, color=C_CON, alpha=.85)
ax[1].set(title='log(Ventas FEMSA) — simétrica', xlabel='log(COP / mes)', ylabel='tiendas')
plt.tight_layout(); plt.show()

> **Hallazgo 1 — las ventas son log-normales.** En niveles la distribución es muy asimétrica
> (asimetría ≈ 4,4); en logaritmo es prácticamente simétrica (≈ 0,0). Esto tiene dos consecuencias que
> usamos en todo el cuaderno:
>
> - El **modelo de ventas** (módulo 3) se entrena sobre `log(ventas)`: los errores se vuelven homocedásticos
>   y una tienda gigante deja de dominar el ajuste.
> - El **efecto de la nevera** (módulo 4) se comporta como un **porcentaje**, no como un monto fijo. Una
>   nevera que sube las ventas un 30 % vale COP 90.000 en una tienda de 300.000 y COP 900.000 en una de
>   3.000.000. Esa es la razón por la que la selección de tiendas importa tanto.

In [ ]:
# ----------------------------- 1.3 El parque de neveras -----------------------------
fig, ax = plt.subplots(1, 3, figsize=(13.5, 4))

pres = df['presencia_neveras'].value_counts()
ax[0].bar(pres.index, pres.values, color=[C_CON,'#4c9f70',C_SIN,'#9aa0a6'])
ax[0].set(title='Presencia de neveras', ylabel='tiendas'); ax[0].tick_params(axis='x', rotation=20)
for i, val in enumerate(pres.values):
    ax[0].text(i, val, f'{val:,}'.replace(',','.'), ha='center', va='bottom', fontsize=8)

cob = pd.DataFrame({'FEMSA':[df.nevera_femsa.mean()], 'Postobón':[df.nevera_postobon.mean()],
                    'Propia':[df.nevera_propia_tienda.mean()], 'Otras marcas':[df.nevera_otras_marcas.mean()]}).T[0]
ax[1].barh(cob.index, cob.values*100, color=[C_CON,C_SIN,'#9aa0a6','#c9ada7'])
ax[1].set(title='Cobertura de neveras (% de tiendas)', xlabel='%')
for i, val in enumerate(cob.values*100):
    ax[1].text(val, i, f' {val:.1f}%', va='center', fontsize=8)

t = df[df.nevera_femsa == 1]
ax[2].hist(t.antiguedad_nevera_femsa_anios, bins=30, color=C_CON, alpha=.85)
ax[2].set(title='Antigüedad de la nevera FEMSA instalada', xlabel='años')
plt.tight_layout(); plt.show()

print(f'Tiendas con nevera FEMSA : {df.nevera_femsa.sum():,}  ({df.nevera_femsa.mean():.1%})')
print(f'Tiendas sin nevera FEMSA : {(df.nevera_femsa==0).sum():,}  ({(df.nevera_femsa==0).mean():.1%})')
print('\nVentas FEMSA promedio según configuración de la nevera instalada:')
print(t.groupby('puertas_nevera_femsa')['ventas_femsa_cop_mes'].agg(['mean','count'])
       .rename(columns={'mean':'ventas_prom','count':'tiendas'}).round(0).to_string())
print('\nSegún si la nevera funciona:')
print(t.groupby('nevera_femsa_operativa')['ventas_femsa_cop_mes'].agg(['mean','count'])
       .rename(columns={'mean':'ventas_prom','count':'tiendas'}).round(0).to_string())

> **Hallazgo 2 — la "nevera" no es una sola cosa.** Entre las tiendas tratadas hay neveras de 1, 2 y 4
> puertas, de 0 a 15 años, y unas 3.800 que **no funcionan**. Las ventas crecen con el número de puertas y
> caen cuando la nevera no es operativa. Como FEMSA va a instalar **una nevera nueva y operativa**, en el
> módulo 4 definimos el tratamiento con precisión y no mezclamos todas las configuraciones.

In [ ]:
# ----------------------------- 1.4 Tiendas con nevera vs sin nevera -----------------------------
# Esta es la tabla mas importante del EDA: muestra que las neveras NO se asignaron al azar.
def dme(col):
    'Diferencia de medias estandarizada (SMD) entre tiendas con y sin nevera FEMSA.'
    a, b = df.loc[df.nevera_femsa == 1, col], df.loc[df.nevera_femsa == 0, col]
    s = np.sqrt((a.var() + b.var()) / 2)
    return a.mean(), b.mean(), (a.mean() - b.mean()) / s if s > 0 else 0.0

comp_vars = ['ventas_femsa_cop_mes','clientes_prom_dia','area_m2','n_referencias_total','poblacion_500m',
             'num_empleados','horas_abre_dia','estrato','visitas_mes_femsa','temperatura_media_c',
             'n_tiendas_competencia_500m','antiguedad_anios','nevera_propia_tienda','nevera_postobon',
             'tiene_datafono','pct_quiebre_femsa']
tabla = pd.DataFrame([dict(zip(['variable','con_nevera','sin_nevera','SMD'], (c,)+dme(c))) for c in comp_vars])
tabla['SMD_abs'] = tabla.SMD.abs()
tabla = tabla.sort_values('SMD_abs', ascending=False)
print('Tiendas CON nevera FEMSA vs SIN nevera FEMSA\n')
print(tabla[['variable','con_nevera','sin_nevera','SMD']].to_string(index=False,
      formatters={'con_nevera':'{:,.2f}'.format,'sin_nevera':'{:,.2f}'.format,'SMD':'{:+.3f}'.format}))

d_naive = (df.loc[df.nevera_femsa==1,'ventas_femsa_cop_mes'].mean()
           - df.loc[df.nevera_femsa==0,'ventas_femsa_cop_mes'].mean())
print(f'\nDiferencia simple de ventas promedio: {cop(d_naive)} / mes')
print(f'Si fuera causal, la ganancia por tienda seria {cop(MARGEN*d_naive - COSTO_NEVERA)} / mes.')

fig, ax = plt.subplots(figsize=(8, 5))
t2 = tabla.sort_values('SMD_abs')
ax.barh(t2.variable, t2.SMD_abs, color=np.where(t2.SMD_abs > .1, C_SIN, '#9aa0a6'))
ax.axvline(.1, ls='--', c='k', lw=1)
ax.text(.105, 0, ' umbral 0,1', fontsize=8)
ax.set(title='Desbalance entre tiendas con y sin nevera (|SMD|)', xlabel='|SMD|')
plt.tight_layout(); plt.show()

> **Hallazgo 3 — el sesgo de selección es grande y medible.** Las tiendas que ya tienen nevera FEMSA
> reciben **+29 % más clientes**, son **~28 % más grandes** y tienen más referencias en el surtido. Varias
> variables superan el umbral habitual de |SMD| > 0,1, que es el que se usa para declarar un desbalance
> preocupante.
>
> La diferencia simple de ventas (≈ COP 748.000) **no es el efecto de la nevera**: mezcla el efecto real con
> el hecho de que FEMSA puso sus neveras en las mejores tiendas. Tomar ese número como causal llevaría a
> estimar una ganancia de ~COP 127.000 por tienda y a declarar rentables casi todas las tiendas. El módulo 4
> corrige exactamente esto.
>
> Nota útil: `temperatura_media_c` tiene SMD ≈ 0,00 — **no** predice quién recibió nevera, pero más adelante
> veremos que sí determina **cuánto sirve** la nevera. Es una variable de heterogeneidad pura.

In [ ]:
# ----------------------------- 1.5 Donde parece servir mas una nevera -----------------------------
# Comparamos ventas con/sin nevera dentro de grupos homogeneos: una primera pista de heterogeneidad.
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

df['_temp_bin'] = pd.cut(df.temperatura_media_c, [0,16,20,24,28,40],
                         labels=['<16°','16-20°','20-24°','24-28°','>28°'])
g = df.groupby(['_temp_bin','nevera_femsa'], observed=True)['ventas_femsa_cop_mes'].mean().unstack()/1e6
g.plot(kind='bar', ax=ax[0], color=[C_SIN, C_CON], width=.8)
ax[0].set(title='Ventas por temperatura', xlabel='temperatura media', ylabel='MM COP / mes')
ax[0].legend(['sin nevera','con nevera'], fontsize=8); ax[0].tick_params(axis='x', rotation=0)

df['_traf_bin'] = pd.qcut(df.clientes_prom_dia, 5, labels=['Q1','Q2','Q3','Q4','Q5'])
g = df.groupby(['_traf_bin','nevera_femsa'], observed=True)['ventas_femsa_cop_mes'].mean().unstack()/1e6
g.plot(kind='bar', ax=ax[1], color=[C_SIN, C_CON], width=.8)
ax[1].set(title='Ventas por tráfico (quintiles)', xlabel='clientes/día', ylabel='')
ax[1].legend(['sin nevera','con nevera'], fontsize=8); ax[1].tick_params(axis='x', rotation=0)

g = (df.groupby(['region','nevera_femsa'])['ventas_femsa_cop_mes'].mean().unstack()/1e6)
g['brecha'] = g[1] - g[0]
g = g.sort_values('brecha')
ax[2].barh(g.index, g.brecha, color=C_ML)
ax[2].set(title='Brecha de ventas con/sin nevera por región', xlabel='MM COP / mes')
plt.tight_layout(); plt.show()

print('Brecha de ventas (con − sin nevera) por región, en COP/mes:')
print((g.brecha*1e6).round(0).sort_values(ascending=False).to_string())
print('\nTemperatura media por región:')
print(df.groupby('region').temperatura_media_c.mean().round(1).sort_values(ascending=False).to_string())
df.drop(columns=['_temp_bin','_traf_bin'], inplace=True)

> **Hallazgo 4 — el calor y el tráfico mandan.** La brecha de ventas entre tiendas con y sin nevera crece
> fuertemente con la temperatura y con el tráfico. Caribe (≈ 28 °C de media) muestra la brecha más grande y
> Eje Cafetero la más pequeña. Esto es intuitivo —una bebida fría vale más donde hace calor y donde pasa
> mucha gente— y anticipa el resultado del módulo 4: **el efecto es heterogéneo**, así que la lista de
> 1.000 tiendas no debe ser simplemente "las más grandes".
>
> Ojo: estas brechas todavía están contaminadas por el sesgo de selección. Son una pista, no una estimación.

In [ ]:
# ----------------------------- 1.6 Tiendas elegibles -----------------------------
df['elegible'] = (df.nevera_femsa == 0) & ~((df.nevera_postobon == 1) & (df.area_m2 < AREA_MIN_SI_POSTOBON))

n_sin   = int((df.nevera_femsa == 0).sum())
n_excl  = int(((df.nevera_femsa == 0) & (df.nevera_postobon == 1) & (df.area_m2 < AREA_MIN_SI_POSTOBON)).sum())
n_elig  = int(df.elegible.sum())
print('Embudo de elegibilidad')
print(f'  Tiendas en la base                         {len(df):>7,}')
print(f'  Con nevera FEMSA (no elegibles)           -{int(df.nevera_femsa.sum()):>7,}')
print(f'  Sin nevera FEMSA                           {n_sin:>7,}')
print(f'  Nevera Postobón y < 12 m² (no cabe otra)  -{n_excl:>7,}')
print(f'  ELEGIBLES                                  {n_elig:>7,}')
print(f'\nSe instalan {N_NEVERAS:,} neveras => solo el {N_NEVERAS/n_elig:.1%} de las elegibles la recibe.')
print('La restricción es fuerte: hay que ordenar bien, no solo acertar el signo.\n')

E = df[df.elegible]
print('Perfil de las elegibles frente al resto de la base:')
perfil = pd.DataFrame({'elegibles': E[comp_vars[:12]].mean(), 'toda_la_base': df[comp_vars[:12]].mean()})
print(perfil.round(2).to_string())

print(f'\nTasa de cambio de proveedor en los próximos 6 meses: {df.cambio_proveedor_principal_6m.mean():.2%}')
print(f'  entre elegibles: {E.cambio_proveedor_principal_6m.mean():.2%}')
print('  => clase muy desbalanceada; el módulo 5 lo trata explícitamente.')

> **Hallazgo 5 — hay 12.700 tiendas elegibles para 1.000 neveras.** Solo el 7,9 % de las candidatas recibe
> nevera. Con esa restricción, lo que decide la ganancia no es acertar el promedio del efecto sino
> **ordenar bien** las 12.700 tiendas. Un modelo que estime bien el efecto promedio pero ordene mal
> desperdicia dinero.
>
> Además, las elegibles son en promedio más pequeñas que las tiendas ya atendidas (885.000 vs 1.629.000 COP
> de venta mensual), así que no podemos extrapolar sobre ellas el efecto medido en las grandes sin cuidado.

---
# Módulo 2 — Preparación de datos y control de fuga

Aquí se decide **qué variables puede ver cada modelo**. Es el módulo que más silenciosamente arruina un
proyecto: si dejamos entrar una columna que contiene la respuesta, el modelo saca R² = 1 y no sirve para nada.

Distinguimos **tres** conjuntos de variables, porque los tres modelos del proyecto responden preguntas
distintas y por lo tanto tienen derecho a ver cosas distintas:

| Conjunto | Para qué | Qué excluye |
|---|---|---|
| `X_VENTAS` | Módulo 3: predecir `ventas_femsa_cop_mes` | Todo lo que *contiene* la respuesta |
| `X_CAUSAL` | Módulo 4: efecto de la nevera | Lo anterior **+** todo lo posterior al tratamiento |
| `X_CHURN`  | Módulo 5: predecir cambio de proveedor | Nada (todo es previo a la ventana de 6 meses) |

In [ ]:
# ----------------------------- 2.1 Fuga directa: verificar las identidades -----------------------------
# No basta sospechar: comprobamos numericamente que ciertas columnas SON la respuesta.
SKU_FEMSA = ['cocacola_400ml','cocacola_1500ml','cocacola_ret_1250ml','cocacola_sinazucar_400ml',
             'quatro_400ml','sprite_400ml','brisa_600ml','delvalle_fresh_400ml','powerade_500ml','fuze_tea_400ml']
SKU_POSTOBON = ['colombiana_400ml','manzana_400ml','manzana_1500ml','pepsi_400ml','sevenup_400ml',
                'cristal_600ml','hit_500ml','mr_tea_500ml','squash_500ml','speedmax_250ml']

identidades = {
 'sum(cop_<sku FEMSA>) == ventas_femsa_cop_mes':
    np.allclose(df[['cop_'+s for s in SKU_FEMSA]].sum(axis=1), df.ventas_femsa_cop_mes),
 'sum(u_<sku FEMSA>) == unidades_femsa_mes':
    np.allclose(df[['u_'+s for s in SKU_FEMSA]].sum(axis=1), df.unidades_femsa_mes),
 'ventas_femsa + ventas_postobon == ventas_bebidas':
    np.allclose(df.ventas_femsa_cop_mes + df.ventas_postobon_cop_mes, df.ventas_bebidas_cop_mes),
 'ventas_bebidas + ventas_otras == ventas_totales':
    np.allclose(df.ventas_bebidas_cop_mes + df.ventas_otras_categorias_cop_mes, df.ventas_totales_tienda_cop_mes),
 'share_femsa_valor == ventas_femsa / ventas_bebidas':
    np.allclose(df.share_femsa_valor, df.ventas_femsa_cop_mes/df.ventas_bebidas_cop_mes, atol=1e-3),
}
print('Identidades contables verificadas sobre las 38.000 filas:\n')
for k, v in identidades.items():
    print(('  CONFIRMADA  ' if v else '  no se cumple') + f' | {k}')
print('\nConclusión: las columnas u_*, cop_* y los agregados de ventas no son "variables predictoras":')
print('son la respuesta escrita de otra forma. Usarlas daria R2 ~ 1 y un modelo inservible.')

In [ ]:
# ----------------------------- 2.2 Definicion de los conjuntos de variables -----------------------------
COLS_SKU = [c for c in df.columns if c.startswith(('u_','cop_'))]          # 40 columnas de SKU

# (a) Fuga directa del objetivo de ventas: contiene ventas_femsa por construccion
FUGA_DIRECTA = COLS_SKU + ['unidades_femsa_mes','ventas_femsa_cop_mes','ventas_bebidas_cop_mes',
                           'share_femsa_valor','n_skus_activos_femsa','ventas_totales_tienda_cop_mes',
                           'ticket_promedio_cop','ventas_por_m2_cop','peso_bebidas_en_ventas',
                           'proveedor_principal']

# (b) Resultados del MISMO mes que no contienen la respuesta pero son simultaneos a ella.
#     No son causa de las ventas de FEMSA: son consecuencia de las mismas decisiones de la tienda.
#     Incluirlos infla el R2 sin aportar nada accionable y contamina el analisis causal.
RESULTADOS_SIMULTANEOS = ['unidades_postobon_mes','ventas_postobon_cop_mes','n_skus_activos_postobon',
                          'ventas_otras_categorias_cop_mes']

# (c) Variables que describen la nevera FEMSA: son el tratamiento o funcion directa de el.
#     En el modulo 4 deben salir TODAS, o el modelo "explicaria" el efecto con ellas mismas.
COLS_TRATAMIENTO = ['nevera_femsa','num_neveras_femsa','puertas_nevera_femsa',
                    'antiguedad_nevera_femsa_anios','nevera_femsa_operativa','presencia_neveras']

# (d) Identificadores y objetivos
NO_PREDICTORAS = ['id_tienda','cambio_proveedor_principal_6m','elegible']

# --- Conjunto para el MODULO 3 (modelo de ventas) ---
# Si incluye las variables de nevera: describe la realidad actual (cuanto vende una tienda tal como esta).
X_VENTAS = [c for c in df.columns if c not in FUGA_DIRECTA + RESULTADOS_SIMULTANEOS + NO_PREDICTORAS]

# --- Conjunto para el MODULO 4 (efecto causal) ---
# Solo variables PREVIAS o ajenas a la decision de instalar la nevera FEMSA.
X_CAUSAL = [c for c in X_VENTAS if c not in COLS_TRATAMIENTO]

# --- Conjunto para el MODULO 5 (riesgo de cambio) ---
# El objetivo ocurre en los 6 meses SIGUIENTES: toda la foto de hoy es informacion legitima.
X_CHURN = [c for c in df.columns if c not in ['id_tienda','cambio_proveedor_principal_6m','elegible']]

print(f'Columnas totales en la base         : {df.shape[1]}')
print(f'  excluidas por fuga directa        : {len(FUGA_DIRECTA)}')
print(f'  excluidas por simultaneidad       : {len(RESULTADOS_SIMULTANEOS)}')
print(f'  identificadores y objetivos       : {len(NO_PREDICTORAS)}')
print()
print(f'X_VENTAS (modulo 3)  : {len(X_VENTAS):>3} variables')
print(f'X_CAUSAL (modulo 4)  : {len(X_CAUSAL):>3} variables  (= X_VENTAS - {len(COLS_TRATAMIENTO)} de nevera FEMSA)')
print(f'X_CHURN  (modulo 5)  : {len(X_CHURN):>3} variables')
print('\nVariables de X_CAUSAL:')
print(textwrap.fill(', '.join(X_CAUSAL), 100))

### Por qué cada exclusión

**(a) Fuga directa — 50 columnas.** `ventas_femsa_cop_mes` es exactamente la suma de las 10 columnas
`cop_<sku FEMSA>`, verificado arriba. Por la misma vía entran `ventas_bebidas_cop_mes`,
`ventas_totales_tienda_cop_mes`, `ticket_promedio_cop`, `ventas_por_m2_cop`, `peso_bebidas_en_ventas`,
`share_femsa_valor` y `proveedor_principal`: todas se calculan *usando* la respuesta. Dejar cualquiera de
ellas produce un R² cercano a 1 que no se puede usar para decidir nada, porque para una tienda nueva no
conoceríamos esos valores.

**(b) Resultados simultáneos — 4 columnas.** Las ventas de Postobón del mismo mes no contienen la respuesta,
pero tampoco la *causan*: son el resultado paralelo de las mismas condiciones de la tienda. En el análisis
causal son peores aún: una nevera de FEMSA puede **quitarle** ventas a Postobón, así que
`ventas_postobon_cop_mes` está afectada por el tratamiento. Condicionar sobre una variable afectada por el
tratamiento sesga el efecto estimado (es un *colisionador*). Las excluimos de los dos módulos.

**(c) Variables del tratamiento — 6 columnas.** `num_neveras_femsa`, `puertas_nevera_femsa`,
`antiguedad_nevera_femsa_anios`, `nevera_femsa_operativa` y `presencia_neveras` son funciones determinísticas
de `nevera_femsa`: valen 0 (o "Ninguna"/"Solo Postobón") en todas las tiendas sin nevera. Si el modelo causal
las viera, podría reproducir el tratamiento a partir de ellas y el efecto estimado se volvería incoherente.
Se usan en el módulo 3 (donde solo describimos la realidad) y se excluyen en el módulo 4.

**Lo que sí conservamos en `X_CAUSAL`:** las variables de nevera de **Postobón** y las de nevera propia /
otras marcas. Son competencia por el espacio frío y están determinadas antes de nuestra decisión, así que son
confusores legítimos que *debemos* controlar, no fuga.

In [ ]:
# ----------------------------- 2.3 Codificacion -----------------------------
# Dos codificaciones segun la familia de modelo:
#   - arboles (RandomForest, HistGradientBoosting): codigos enteros + declaracion de categoricas
#   - lineales (Ridge): one-hot, porque un codigo entero impondria un orden falso
CATEGORICAS = list(df[X_VENTAS].select_dtypes(include=['object','string','category']).columns)
NUMERICAS   = [c for c in X_VENTAS if c not in CATEGORICAS]
print(f'Categóricas ({len(CATEGORICAS)}): {CATEGORICAS}')
print(f'Numéricas   ({len(NUMERICAS)})')
print('\nCardinalidad de las categóricas:')
for c in CATEGORICAS:
    print(f'  {c:14s} {df[c].nunique():>3} niveles')

# Mapas de categorias fijos (reproducibilidad: el mismo codigo en cualquier corrida)
MAPAS_CAT = {c: pd.CategoricalDtype(categories=sorted(df[c].unique()), ordered=False) for c in CATEGORICAS}

def codificar_arboles(datos, columnas):
    'Devuelve X numerico (codigos enteros) y los indices de las columnas categoricas.'
    X = datos[columnas].copy()
    cats = [c for c in columnas if c in MAPAS_CAT]
    for c in cats:
        X[c] = X[c].astype(MAPAS_CAT[c]).cat.codes.astype(float)
    return X, [X.columns.get_loc(c) for c in cats]

X_arb, IDX_CAT = codificar_arboles(df, X_VENTAS)
print(f'\nMatriz para árboles: {X_arb.shape}, categóricas en posiciones {IDX_CAT}')
print('Sin valores faltantes:', int(X_arb.isna().sum().sum()) == 0)

In [ ]:
# ----------------------------- 2.4 Particion de datos -----------------------------
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate, cross_val_predict

# Particion 80/20 estratificada por tratamiento y por tramo de ventas, para que train y test
# tengan la misma mezcla de tiendas grandes/pequenas y con/sin nevera.
estrato_part = (df.nevera_femsa.astype(str) + '_' +
                pd.qcut(df.ventas_femsa_cop_mes, 5, labels=False).astype(str))
idx_train, idx_test = train_test_split(df.index.values, test_size=0.20,
                                       random_state=SEED, stratify=estrato_part)
df['particion'] = np.where(df.index.isin(idx_train), 'train', 'test')

CV_REG = KFold(5, shuffle=True, random_state=SEED)             # para modelos de regresion
CV_CLF = StratifiedKFold(5, shuffle=True, random_state=SEED)   # para clasificacion

print(f'Train: {len(idx_train):,} tiendas   Test: {len(idx_test):,} tiendas')
print('\nComprobación del balance de la partición:')
chk = df.groupby('particion').agg(
    nevera_femsa=('nevera_femsa','mean'),
    ventas_media=('ventas_femsa_cop_mes','mean'),
    elegibles=('elegible','mean'),
    tasa_cambio=('cambio_proveedor_principal_6m','mean'))
print(chk.round(4).to_string())
print('\nValidación cruzada: 5 folds, shuffle, semilla', SEED)

**Nota sobre el uso de la partición.** Para la comparación de modelos del módulo 3 usamos validación cruzada
de 5 *folds* sobre `train` (así cada modelo se evalúa con 5 estimaciones y podemos ver la dispersión) y
reservamos `test` como prueba final de que no sobreajustamos la selección de modelo.

Para el módulo 4 el enfoque es distinto: no hay "respuesta correcta" observable para el efecto causal, así
que no sirve un *hold-out* clásico. Allí usamos **cross-fitting** (muestras cruzadas): cada tienda recibe
predicciones de modelos que nunca la vieron. Se explica en su sección.

---
# Módulo 3 — Modelo de ventas

**Objetivo:** predecir `ventas_femsa_cop_mes` con al menos tres algoritmos, incluido un modelo base lineal,
compararlos con validación cruzada e interpretar las variables más importantes.

**Qué pregunta responde este modelo:** *¿cuánto vende una tienda, dadas sus características actuales?*
Es un modelo **descriptivo/predictivo**, no causal: aquí sí puede ver las variables de nevera, porque
describe la realidad tal como está. No lo usamos para decidir dónde poner neveras —para eso está el
módulo 4— pero nos sirve para tres cosas:

1. Verificar que las variables de la base realmente explican las ventas (si no, no habría señal).
2. Saber qué variables mueven las ventas, como insumo del tablero.
3. Fijar el techo de desempeño predictivo contra el cual comparar los modelos del módulo 4.

**Objetivo en logaritmo.** Entrenamos sobre `log(ventas)` por el Hallazgo 1: en niveles, el 1 % de tiendas
más grandes domina el error cuadrático y el modelo se vuelve malo para el 99 % restante. Reportamos las
métricas en las dos escalas: en log (para comparar modelos) y en COP (para interpretar el error).

In [ ]:
# ----------------------------- 3.1 Preparacion de las matrices -----------------------------
from sklearn.linear_model import RidgeCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.inspection import permutation_importance

y_log = np.log(df['ventas_femsa_cop_mes'].values)        # objetivo en logaritmo
y_cop = df['ventas_femsa_cop_mes'].values.astype(float)  # objetivo en COP

Xv_arb, IDXC = codificar_arboles(df, X_VENTAS)   # matriz para arboles
Xv_lin = df[X_VENTAS].copy()                     # DataFrame crudo para el pipeline lineal

tr, te = idx_train, idx_test
print(f'X_VENTAS: {len(X_VENTAS)} variables | train {len(tr):,} | test {len(te):,}')

# Preprocesador del modelo lineal: estandarizar numericas + one-hot a categoricas
prep_lineal = ColumnTransformer([
    ('num', StandardScaler(), NUMERICAS),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAS)])

MODELOS = {
 'Lineal (Ridge)'      : ('lin', Pipeline([('prep', prep_lineal),
                                           ('mod', RidgeCV(alphas=np.logspace(-2, 3, 20)))])),
 'Árbol de decisión'   : ('arb', DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, random_state=SEED)),
 'Random Forest'       : ('arb', RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                                      max_features=0.4, random_state=SEED, n_jobs=-1)),
 'Gradient Boosting'   : ('arb', HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06,
                                                      max_leaf_nodes=31, min_samples_leaf=20,
                                                      l2_regularization=1.0,
                                                      categorical_features=IDXC, random_state=SEED)),
}
print('Modelos a comparar:', ', '.join(MODELOS))

In [ ]:
# ----------------------------- 3.2 Validacion cruzada 5-fold sobre TRAIN -----------------------------
# Comparamos los modelos con la misma particion de folds (misma semilla) para que la comparacion sea valida.
filas = []
pred_oof = {}
for nombre, (tipo, modelo) in MODELOS.items():
    X = Xv_arb if tipo == 'arb' else Xv_lin
    cv_res = cross_validate(modelo, X.iloc[tr], y_log[tr], cv=CV_REG,
                            scoring=['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error'],
                            n_jobs=1, return_train_score=True)
    # predicciones fuera de fold, para graficos y para el error en COP
    oof = cross_val_predict(modelo, X.iloc[tr], y_log[tr], cv=CV_REG, n_jobs=1)
    pred_oof[nombre] = oof
    smear = np.exp(y_log[tr] - oof).mean()              # correccion de Duan al volver a COP
    filas.append({'modelo': nombre,
                  'R2_cv': cv_res['test_r2'].mean(), 'R2_sd': cv_res['test_r2'].std(),
                  'R2_train': cv_res['train_r2'].mean(),
                  'RMSE_log': -cv_res['test_neg_root_mean_squared_error'].mean(),
                  'MAE_log': -cv_res['test_neg_mean_absolute_error'].mean(),
                  'MAE_COP': mean_absolute_error(y_cop[tr], np.exp(oof) * smear),
                  'MAPE_%': (np.abs(np.exp(oof)*smear - y_cop[tr]) / y_cop[tr]).mean() * 100})
    print(f'  {nombre:22s} listo  (R2 cv = {cv_res["test_r2"].mean():.4f})')

comp = pd.DataFrame(filas).sort_values('R2_cv', ascending=False).reset_index(drop=True)
print('\n===== COMPARACIÓN POR VALIDACIÓN CRUZADA (5 folds, sobre train) =====\n')
print(comp.to_string(index=False, formatters={
    'R2_cv':'{:.4f}'.format, 'R2_sd':'{:.4f}'.format, 'R2_train':'{:.4f}'.format,
    'RMSE_log':'{:.4f}'.format, 'MAE_log':'{:.4f}'.format,
    'MAE_COP':'{:,.0f}'.format, 'MAPE_%':'{:.1f}'.format}))

In [ ]:
# ----------------------------- 3.3 Grafico de la comparacion -----------------------------
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
o = comp.sort_values('R2_cv')
col = [C_ML if m == comp.modelo.iloc[0] else '#9aa0a6' for m in o.modelo]
ax[0].barh(o.modelo, o.R2_cv, xerr=o.R2_sd, color=col, capsize=3)
ax[0].set(title='R² en validación cruzada (escala log)', xlabel='R²', xlim=(0, .8))
for i, (v, s) in enumerate(zip(o.R2_cv, o.R2_sd)):
    ax[0].text(v + s + .01, i, f'{v:.3f}', va='center', fontsize=8)

mejor = comp.modelo.iloc[0]
ax[1].scatter(y_log[tr], pred_oof[mejor], s=3, alpha=.08, color=C_CON)
lims = [y_log[tr].min(), y_log[tr].max()]
ax[1].plot(lims, lims, 'k--', lw=1)
ax[1].set(title=f'{mejor}: predicho vs real (fuera de fold)',
          xlabel='log(ventas) real', ylabel='log(ventas) predicho')
plt.tight_layout(); plt.show()

print(f'Mejor modelo por R² de validación cruzada: {mejor}')
print(f'\nSobreajuste (R² train − R² cv):')
for _, r in comp.iterrows():
    print(f'  {r.modelo:22s} {r.R2_train - r.R2_cv:+.4f}')

In [ ]:
# ----------------------------- 3.4 Evaluacion final en TEST -----------------------------
# El test no se uso para elegir modelo: es la prueba honesta de desempeno.
res_test = []
MODELO_VENTAS = None
for nombre, (tipo, modelo) in MODELOS.items():
    X = Xv_arb if tipo == 'arb' else Xv_lin
    m = modelo.fit(X.iloc[tr], y_log[tr])
    p = m.predict(X.iloc[te])
    smear = np.exp(y_log[tr] - m.predict(X.iloc[tr])).mean()
    res_test.append({'modelo': nombre,
                     'R2_test': r2_score(y_log[te], p),
                     'RMSE_log_test': root_mean_squared_error(y_log[te], p),
                     'MAE_COP_test': mean_absolute_error(y_cop[te], np.exp(p)*smear)})
    if nombre == mejor:
        MODELO_VENTAS, SMEAR_VENTAS = m, smear

test_tab = pd.DataFrame(res_test).sort_values('R2_test', ascending=False)
print('===== DESEMPEÑO EN TEST (20 % reservado) =====\n')
print(test_tab.to_string(index=False, formatters={
    'R2_test':'{:.4f}'.format, 'RMSE_log_test':'{:.4f}'.format, 'MAE_COP_test':'{:,.0f}'.format}))

mejor_r2 = test_tab.R2_test.iloc[0]
base_r2  = test_tab.loc[test_tab.modelo == 'Lineal (Ridge)', 'R2_test'].iloc[0]
print(f'\nGanancia del mejor modelo sobre el base lineal: {mejor_r2 - base_r2:+.4f} de R²'
      f'  ({(mejor_r2-base_r2)/base_r2:+.1%} relativo)')

In [ ]:
# ----------------------------- 3.5 Importancia de variables -----------------------------
# Permutation importance: cuanto empeora el R2 del TEST al barajar cada variable.
# Es comparable entre modelos y no premia a las variables de alta cardinalidad.
sub = np.random.RandomState(SEED).choice(te, size=min(6000, len(te)), replace=False)
pi = permutation_importance(MODELO_VENTAS, Xv_arb.loc[sub], y_log[sub],
                            n_repeats=5, random_state=SEED, scoring='r2', n_jobs=1)
imp = (pd.DataFrame({'variable': X_VENTAS, 'importancia': pi.importances_mean,
                     'sd': pi.importances_std})
         .sort_values('importancia', ascending=False).reset_index(drop=True))

print(f'Importancia por permutación — {mejor} (caída de R² en test)\n')
print(imp.head(18).to_string(index=False, formatters={'importancia':'{:.4f}'.format,'sd':'{:.4f}'.format}))

fig, ax = plt.subplots(figsize=(8, 6))
top = imp.head(18).sort_values('importancia')
colores = [C_CON if 'nevera' in v else C_ML if v in ('temperatura_media_c','clientes_prom_dia') else '#9aa0a6'
           for v in top.variable]
ax.barh(top.variable, top.importancia, xerr=top.sd, color=colores, capsize=2)
ax.set(title=f'Variables más importantes — {mejor}', xlabel='caída de R² al permutar')
plt.tight_layout(); plt.show()

In [ ]:
# ----------------------------- 3.6 Interpretacion: coeficientes del modelo lineal -----------------------------
# El modelo lineal en log da una lectura directa: el coeficiente es ~% de cambio en ventas.
lin = MODELOS['Lineal (Ridge)'][1].fit(Xv_lin.iloc[tr], y_log[tr])
nombres = (NUMERICAS + list(lin.named_steps['prep'].named_transformers_['cat']
                            .get_feature_names_out(CATEGORICAS)))
coef = pd.DataFrame({'variable': nombres, 'coef': lin.named_steps['mod'].coef_})
coef['efecto_%'] = (np.exp(coef.coef) - 1) * 100
coef['abs'] = coef.coef.abs()

print('Coeficientes del modelo lineal (variables numéricas estandarizadas:')
print('el efecto es el cambio en ventas al subir 1 desviación estándar)\n')
print(coef[coef.variable.isin(NUMERICAS)].nlargest(12, 'abs')[['variable','coef','efecto_%']]
      .to_string(index=False, formatters={'coef':'{:+.4f}'.format,'efecto_%':'{:+.1f}%'.format}))
print('\nCategóricas con mayor efecto:')
print(coef[~coef.variable.isin(NUMERICAS)].nlargest(8, 'abs')[['variable','coef','efecto_%']]
      .to_string(index=False, formatters={'coef':'{:+.4f}'.format,'efecto_%':'{:+.1f}%'.format}))

coef.drop(columns='abs').to_csv(f'{RUTA_SALIDAS}/coeficientes_modelo_lineal.csv', index=False)
imp.to_csv(f'{RUTA_SALIDAS}/importancia_variables.csv', index=False)
comp.merge(test_tab, on='modelo').to_csv(f'{RUTA_SALIDAS}/metricas_modelos_ventas.csv', index=False)
print(f'\nGuardados: importancia_variables.csv, metricas_modelos_ventas.csv, coeficientes_modelo_lineal.csv')

### Lectura del módulo 3

**Comparación de modelos.** En test, el modelo base lineal explica el **66,1 %** de la varianza de
`log(ventas)`, el Random Forest el **69,1 %** y el *Gradient Boosting* el **70,6 %**. La mejora del boosting
sobre el lineal es real pero modesta (**+4,5 puntos de R²**, +6,8 % relativo): **la relación entre las
características de la tienda y sus ventas es en buena medida multiplicativa y suave**, que es precisamente
lo que un modelo lineal en logaritmo captura bien. El árbol de decisión individual queda incluso por debajo
del lineal (65,4 %), lo que confirma que la ganancia del bosque y del boosting viene de promediar muchos
árboles, no de la no linealidad por sí sola.

Vale la pena mirar la columna de sobreajuste: el Random Forest alcanza R² = 0,88 en entrenamiento contra
0,70 en validación cruzada (una brecha de 18 puntos), mientras el boosting se queda en 4,5 puntos y el
lineal en 0,2. El bosque memoriza; por eso lo descartamos aunque su desempeño en test sea aceptable.

El error típico es grande en términos absolutos (MAE ≈ COP 438.000) porque las ventas van de 15.000 a 42
millones. En términos relativos el mejor modelo se equivoca alrededor de un **41 %** en una tienda típica:
suficiente para ordenar tiendas, insuficiente para prometer un pronóstico puntual.

**Variables importantes.** Las que más pesan son las de **escala y tráfico** (`clientes_prom_dia`,
`n_referencias_total`, `area_m2`, `poblacion_500m`, `num_empleados`), seguidas por **temperatura** y por las
**variables de nevera**. El coeficiente lineal de `nevera_femsa` es positivo y grande, pero —y esto es
central para el proyecto— **ese coeficiente no es el efecto causal de instalar una nevera**: está capturando
también que FEMSA puso sus neveras en las mejores tiendas (Hallazgo 3). Interpretarlo como efecto sería
exactamente el error que el módulo 4 corrige.

**Por qué este modelo no decide la lista.** Un modelo de ventas responde "¿cuánto vende esta tienda?". La
pregunta de negocio es "¿cuánto *más* vendería con una nevera?". Son preguntas distintas y la segunda no se
contesta con la primera. Si usáramos este modelo para elegir, escogeríamos las tiendas que más venden, que es
una de las dos reglas simples que el módulo 6 derrota.

---
# Módulo 4 — Efecto causal de la nevera

Este es el módulo que decide el proyecto. Todo lo anterior fue preparación.

## El problema en una frase

La diferencia simple de ventas entre tiendas con y sin nevera es de **≈ COP 748.000 al mes**. Si la
tomáramos como el efecto de la nevera, la ganancia por tienda sería `0,25 × 748.000 − 60.000 ≈ COP 127.000`
y **casi cualquier** tienda elegible parecería rentable. Pero el Hallazgo 3 mostró que las tiendas con nevera
reciben 29 % más clientes y son 28 % más grandes *antes* de cualquier efecto de la nevera. Esa diferencia
mezcla dos cosas:

$$\underbrace{E[Y|T=1] - E[Y|T=0]}_{\text{diferencia simple} \approx 748.000} = \underbrace{\text{efecto de la nevera}}_{\text{lo que queremos}} + \underbrace{\text{sesgo de selección}}_{\text{FEMSA eligió las mejores tiendas}}$$

Necesitamos el primer término solo.

## Qué queremos estimar, exactamente

Para cada tienda elegible $i$ queremos el **efecto individual de tratamiento (CATE)**:

$$\tau(x_i) = E[\,Y_i(1) - Y_i(0) \mid X_i = x_i\,]$$

donde $Y_i(1)$ son las ventas con nevera nueva y $Y_i(0)$ sin ella. Solo observamos una de las dos: el
problema es de datos faltantes, no de predicción.

**No queremos el promedio.** Queremos el *orden*: con 1.000 neveras para 12.700 candidatas, lo que determina
la ganancia es identificar correctamente el 8 % superior.

## Supuestos que debemos asumir (y cómo los revisamos)

| Supuesto | Qué dice | Cómo lo revisamos |
|---|---|---|
| **Ignorabilidad** | Dadas las variables observadas, tener nevera es "como si fuera" aleatorio | No es verificable. Discutimos su plausibilidad y hacemos análisis de sensibilidad |
| **Positividad** | Toda tienda tenía probabilidad > 0 de recibir y de no recibir nevera | Distribución del propensity score (§4.3) |
| **SUTVA** | La nevera de una tienda no afecta las ventas de otra | Discutido en limitaciones |

La ignorabilidad es el supuesto fuerte. Nos apoyamos en que la base trae un conjunto **inusualmente rico** de
determinantes de la asignación: tamaño, tráfico, surtido, entorno, gestión comercial, competencia y
geografía. Son justamente los criterios con los que una fuerza de ventas decide dónde poner una nevera. Aun
así, el módulo cierra con una prueba de cuánto tendría que faltar para invalidar la decisión.

In [ ]:
# ----------------------------- 4.1 Definicion precisa del tratamiento -----------------------------
# FEMSA va a instalar UNA nevera NUEVA y OPERATIVA.
# El grupo tratado de la base mezcla configuraciones: 1 o 2 neveras, 1/2/4 puertas, 0-15 anios,
# y ~3.800 tiendas con la nevera averiada. Comparar contra esa mezcla responde otra pregunta.
print('Composición del grupo con nevera FEMSA:')
tt = df[df.nevera_femsa == 1]
print(pd.crosstab(tt.num_neveras_femsa, tt.nevera_femsa_operativa,
                  rownames=['n° neveras'], colnames=['operativa']).to_string())

# Muestra causal: controles puros + tratadas con UNA nevera OPERATIVA
mask_causal = (df.nevera_femsa == 0) | ((df.num_neveras_femsa == 1) & (df.nevera_femsa_operativa == 1))
dc = df[mask_causal].reset_index(drop=True)

print(f'\nMuestra causal: {len(dc):,} tiendas')
print(f'  control  (sin nevera FEMSA)          {int((dc.nevera_femsa==0).sum()):>7,}')
print(f'  tratadas (1 nevera FEMSA operativa)  {int(dc.nevera_femsa.sum()):>7,}')
print(f'  descartadas de la base               {len(df)-len(dc):>7,}  '
      f'(2 neveras o nevera averiada: otra "dosis")')
print('\nEstimando: efecto de UNA nevera operativa, que es exactamente lo que FEMSA va a instalar.')

In [ ]:
# ----------------------------- 4.2 Matrices del analisis causal -----------------------------
Xc, IDXC_C = codificar_arboles(dc, X_CAUSAL)
T  = dc['nevera_femsa'].values.astype(int)        # tratamiento
Y  = dc['ventas_femsa_cop_mes'].values.astype(float)
LY = np.log(Y)                                     # trabajamos en logaritmo (Hallazgo 1)

from sklearn.ensemble import HistGradientBoostingClassifier
def reg_causal(**kw):
    'Regresor base del analisis causal.'
    return HistGradientBoostingRegressor(random_state=SEED, categorical_features=IDXC_C, **kw)
def clf_causal(**kw):
    'Clasificador base (propensity score).'
    return HistGradientBoostingClassifier(random_state=SEED, categorical_features=IDXC_C, **kw)

print(f'X_CAUSAL: {len(X_CAUSAL)} variables, ninguna posterior al tratamiento')
print(f'Muestra: {len(dc):,} tiendas | tratadas {T.mean():.1%}')

# ---- CROSS-FITTING: cada tienda recibe predicciones de modelos que NO la vieron ----
# Es lo que permite usar modelos flexibles sin que el sobreajuste contamine el efecto estimado.
n = len(dc)
e_oof  = np.zeros(n)   # propensity score  P(T=1|X)
m0_oof = np.zeros(n)   # E[log Y | X, T=0]
m1_oof = np.zeros(n)   # E[log Y | X, T=1]

for k, (i_tr, i_te) in enumerate(CV_CLF.split(Xc, T), 1):
    Xtr, Xte = Xc.iloc[i_tr], Xc.iloc[i_te]
    Ttr, LYtr = T[i_tr], LY[i_tr]
    e_oof[i_te]  = clf_causal().fit(Xtr, Ttr).predict_proba(Xte)[:, 1]
    m0_oof[i_te] = reg_causal().fit(Xtr[Ttr == 0], LYtr[Ttr == 0]).predict(Xte)
    m1_oof[i_te] = reg_causal().fit(Xtr[Ttr == 1], LYtr[Ttr == 1]).predict(Xte)
    print(f'  fold {k}/5 ajustado')
print('\nCross-fitting completo.')

In [ ]:
# ----------------------------- 4.3 Propensity score y positividad -----------------------------
from sklearn.metrics import roc_auc_score
auc_e = roc_auc_score(T, e_oof)
print(f'AUC del propensity score: {auc_e:.4f}')
print('  0,50 = asignación aleatoria   1,00 = asignación perfectamente predecible')
print(f'  => hay selección real, pero moderada: queda mucha variación "casi aleatoria" que explotar.\n')
print(f'Rango del propensity score: [{e_oof.min():.3f}, {e_oof.max():.3f}]')
print(f'  tiendas con e(X) < 0,05 : {(e_oof < .05).sum():,}')
print(f'  tiendas con e(X) > 0,95 : {(e_oof > .95).sum():,}')

LIM_PS = (0.05, 0.95)
en_soporte = (e_oof > LIM_PS[0]) & (e_oof < LIM_PS[1])
print(f'\nTiendas en soporte común: {en_soporte.sum():,} de {n:,} ({en_soporte.mean():.2%})')
print('POSITIVIDAD: se cumple. No hay tiendas con probabilidad ~0 o ~1 de recibir nevera,')
print('asi que para cada tienda existen comparables en el otro grupo. No hace falta recortar la muestra.')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(e_oof[T == 0], bins=50, alpha=.6, label='sin nevera (control)', color=C_SIN, density=True)
ax[0].hist(e_oof[T == 1], bins=50, alpha=.6, label='con nevera (tratada)', color=C_CON, density=True)
ax[0].set(title='Solapamiento del propensity score', xlabel='e(X) = P(nevera | X)', ylabel='densidad')
ax[0].legend(fontsize=8)

# Que variables predicen recibir nevera
pi_e = permutation_importance(clf_causal().fit(Xc, T), Xc, T, n_repeats=3,
                              random_state=SEED, scoring='roc_auc', n_jobs=1)
top_e = pd.DataFrame({'v': X_CAUSAL, 'i': pi_e.importances_mean}).nlargest(10, 'i').sort_values('i')
ax[1].barh(top_e.v, top_e.i, color=C_SIN)
ax[1].set(title='¿Qué determinó dónde puso FEMSA sus neveras?', xlabel='caída de AUC al permutar')
plt.tight_layout(); plt.show()

> **El solapamiento es bueno.** Los dos histogramas se superponen en casi todo el rango y ninguna tienda
> tiene `e(X)` por debajo de 0,05 ni por encima de 0,95. Esto es importante: significa que para cada tienda
> con nevera existen tiendas *parecidas* sin nevera y viceversa, así que el ajuste estadístico interpola
> entre casos comparables en lugar de extrapolar. Si hubiera habido tiendas con `e(X) ≈ 1`, no tendríamos
> contrafactual para ellas y habría que excluirlas.
>
> **AUC = 0,63** dice que la asignación fue selectiva pero lejos de determinista. Las variables que más
> predicen haber recibido nevera son las de **escala del negocio** (tráfico, área, surtido): coherente con
> una fuerza de ventas que prioriza las tiendas grandes.

In [ ]:
# ----------------------------- 4.4 Balance: antes y despues de ponderar -----------------------------
# IPW: ponderamos cada tienda por 1/e(X) si tratada y 1/(1-e(X)) si control.
# Si el propensity score esta bien estimado, la poblacion ponderada queda balanceada.
ec = np.clip(e_oof, LIM_PS[0], LIM_PS[1])
w_ipw = np.where(T == 1, 1/ec, 1/(1-ec))

def smd_pond(col):
    'SMD crudo y SMD despues de ponderar por IPW.'
    v = Xc[col].values.astype(float)
    s = np.sqrt((v[T==1].var() + v[T==0].var())/2)
    if s == 0: return 0., 0.
    crudo = (v[T==1].mean() - v[T==0].mean())/s
    pond  = (np.average(v[T==1], weights=w_ipw[T==1]) -
             np.average(v[T==0], weights=w_ipw[T==0]))/s
    return crudo, pond

bal = pd.DataFrame([{'variable': c, 'SMD_crudo': a, 'SMD_ponderado': b}
                    for c, (a, b) in ((c, smd_pond(c)) for c in X_CAUSAL)])
bal['abs_crudo'] = bal.SMD_crudo.abs(); bal['abs_pond'] = bal.SMD_ponderado.abs()
bal = bal.sort_values('abs_crudo', ascending=False)

print('Balance de covariables — 12 variables más desbalanceadas\n')
print(bal.head(12)[['variable','SMD_crudo','SMD_ponderado']].to_string(index=False,
      formatters={'SMD_crudo':'{:+.3f}'.format,'SMD_ponderado':'{:+.3f}'.format}))
print(f'\nmáx |SMD| antes de ponderar : {bal.abs_crudo.max():.3f}')
print(f'máx |SMD| después de ponderar: {bal.abs_pond.max():.3f}')
print(f'variables con |SMD| > 0,1 antes: {(bal.abs_crudo > .1).sum()} de {len(bal)}')
print(f'variables con |SMD| > 0,1 después: {(bal.abs_pond > .1).sum()} de {len(bal)}')

fig, ax = plt.subplots(figsize=(7.5, 7))
b = bal.head(20).sort_values('abs_crudo')
ax.scatter(b.abs_crudo, b.variable, label='sin ajustar', color=C_SIN, s=45, zorder=3)
ax.scatter(b.abs_pond, b.variable, label='ponderado por IPW', color=C_ML, s=45, zorder=3)
for _, r in b.iterrows():
    ax.plot([r.abs_pond, r.abs_crudo], [r.variable, r.variable], color='#ccc', lw=1.5, zorder=1)
ax.axvline(.1, ls='--', c='k', lw=1); ax.text(.105, -.4, 'umbral 0,1', fontsize=8)
ax.set(title='Love plot: el ajuste elimina el desbalance', xlabel='|SMD|'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
bal.drop(columns=['abs_crudo','abs_pond']).to_csv(f'{RUTA_SALIDAS}/balance_covariables.csv', index=False)

> **El ajuste funciona.** Antes de ponderar, `clientes_prom_dia` tenía un SMD de 0,45 y varias variables
> superaban 0,1. Después de ponderar por el *propensity score*, **ninguna** variable supera 0,05. La
> población ponderada se comporta como si la nevera se hubiera asignado al azar entre tiendas comparables.
>
> Esto valida la *mecánica* del ajuste (el modelo de `e(X)` captura bien la selección observada), aunque no
> puede validar la ignorabilidad: si FEMSA usó un criterio que no está en la base, el balance se vería igual
> de bueno. De ahí la sensibilidad del §4.9.

In [ ]:
# ----------------------------- 4.5 Cinco estimadores del efecto promedio -----------------------------
# Los calculamos todos para mostrar que la correccion no depende de un truco particular.
res = {}

# (1) Diferencia simple de medias: el numero "equivocado" de referencia
res['1. Diferencia simple de medias'] = {
    'log': LY[T==1].mean() - LY[T==0].mean(),
    'cop': Y[T==1].mean() - Y[T==0].mean()}

# (2) Regresion ajustada (S-learner): un modelo con T como variable
Xs = Xc.copy(); Xs['_T'] = T
s_mod = HistGradientBoostingRegressor(random_state=SEED, categorical_features=IDXC_C).fit(Xs, LY)
X1 = Xs.copy(); X1['_T'] = 1
X0 = Xs.copy(); X0['_T'] = 0
res['2. Regresión ajustada (S-learner)'] = {'log': (s_mod.predict(X1) - s_mod.predict(X0)).mean()}

# (3) IPW: ponderacion por probabilidad inversa
res['3. Ponderación IPW'] = {
    'log': np.average(LY[T==1], weights=w_ipw[T==1]) - np.average(LY[T==0], weights=w_ipw[T==0])}

# (4) T-learner: un modelo por grupo (ya calculado con cross-fitting)
res['4. T-learner (dos modelos)'] = {'log': (m1_oof - m0_oof).mean()}

# (5) DR-learner (doblemente robusto): el estimador final
#     Combina los modelos de resultado y el propensity score. Es consistente si CUALQUIERA
#     de los dos esta bien especificado, por eso "doblemente robusto".
psi = (m1_oof - m0_oof
       + T*(LY - m1_oof)/ec
       - (1-T)*(LY - m0_oof)/(1-ec))     # pseudo-resultado por tienda
ate_dr = psi.mean()
se_dr  = psi.std(ddof=1)/np.sqrt(n)
res['5. DR-learner (doble robusto)'] = {'log': ate_dr}

tab = pd.DataFrame([{'estimador': k,
                     'efecto_log': v['log'],
                     'efecto_%': (np.exp(v['log'])-1)*100} for k, v in res.items()])
print('===== EFECTO PROMEDIO DE UNA NEVERA SOBRE LAS VENTAS DE FEMSA =====\n')
print(tab.to_string(index=False, formatters={'efecto_log':'{:+.4f}'.format,'efecto_%':'{:+.1f}%'.format}))
print(f'\nDR-learner: efecto = {ate_dr:+.4f} en log  =>  {np.exp(ate_dr)-1:+.1%} sobre las ventas')
print(f'  error estándar {se_dr:.4f}  |  IC 95% en log: '
      f'[{ate_dr-1.96*se_dr:+.4f}, {ate_dr+1.96*se_dr:+.4f}]')
print(f'  IC 95% en %: [{(np.exp(ate_dr-1.96*se_dr)-1):+.1%}, {(np.exp(ate_dr+1.96*se_dr)-1):+.1%}]')
print(f'\nLa diferencia simple ({(np.exp(res["1. Diferencia simple de medias"]["log"])-1):+.1%}) '
      f'sobreestima el efecto en '
      f'{(np.exp(res["1. Diferencia simple de medias"]["log"])-1)/(np.exp(ate_dr)-1)-1:+.0%}.')

In [ ]:
# ----------------------------- 4.6 CATE: el efecto tienda por tienda -----------------------------
# Regresamos el pseudo-resultado del DR-learner sobre X: la prediccion es el CATE.
# Tambien con cross-fitting, para que el tau de cada tienda venga de un modelo que no la vio.
tau_log_oof = np.zeros(n)
for i_tr, i_te in CV_CLF.split(Xc, T):
    mt = reg_causal(max_iter=300, learning_rate=0.05, min_samples_leaf=60).fit(Xc.iloc[i_tr], psi[i_tr])
    tau_log_oof[i_te] = mt.predict(Xc.iloc[i_te])

# Modelo final sobre toda la muestra causal: sirve para interpretar y para el tablero
MODELO_CATE = reg_causal(max_iter=300, learning_rate=0.05, min_samples_leaf=60).fit(Xc, psi)

dc['tau_log'] = tau_log_oof
print('Efecto individual estimado (en log, = cambio porcentual aproximado):')
for q in [.05,.25,.50,.75,.95]:
    v = np.quantile(tau_log_oof, q)
    print(f'  P{int(q*100):>2}  {v:+.3f}   ({np.exp(v)-1:+.1%} de ventas)')
print(f'\n  tiendas con efecto estimado <= 0: {(tau_log_oof <= 0).sum():,} ({(tau_log_oof<=0).mean():.1%})')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(tau_log_oof, bins=60, color=C_ML, alpha=.85)
ax[0].axvline(ate_dr, color='k', ls='--', lw=1.2)
ax[0].text(ate_dr, ax[0].get_ylim()[1]*.92, f' promedio {ate_dr:.3f}', fontsize=8)
ax[0].set(title='Distribución del efecto individual (CATE)', xlabel='efecto en log(ventas)', ylabel='tiendas')

ax[1].hist((np.exp(tau_log_oof)-1)*100, bins=60, color=C_ML, alpha=.85)
ax[1].set(title='El mismo efecto, en % sobre las ventas', xlabel='% de aumento de ventas')
plt.tight_layout(); plt.show()

> **El efecto es muy heterogéneo.** Va de **+4 %** en el 5 % inferior de tiendas a **+62 %** en el 5 %
> superior, con una mediana de +25,5 % y un promedio de +27,1 %. Solo un 2,2 % de las tiendas tiene efecto
> estimado no positivo. Esta dispersión es el activo del proyecto: si el efecto fuera igual para todas,
> cualquier lista de 1.000 tiendas con ventas altas daría lo mismo y no habría nada que modelar.

In [ ]:
# ----------------------------- 4.7 VALIDACION del modelo de efectos (GATE) -----------------------------
# No observamos el efecto verdadero, asi que no hay R2 posible. Pero si podemos validar el ORDEN:
# agrupamos las tiendas por decil de efecto PREDICHO y estimamos el efecto real de cada decil con
# el estimador doble-robusto. Si el modelo ordena bien, las barras deben crecer de forma monotona.
dec = pd.qcut(tau_log_oof, 10, labels=False)
gate = (pd.DataFrame({'decil': dec+1, 'tau_predicho': tau_log_oof, 'psi': psi})
        .groupby('decil')
        .agg(tau_predicho=('tau_predicho','mean'),
             efecto_DR=('psi','mean'),
             ee=('psi', lambda s: s.std(ddof=1)/np.sqrt(len(s))),
             n=('psi','size')).reset_index())
gate['ic_bajo'] = gate.efecto_DR - 1.96*gate.ee
gate['ic_alto'] = gate.efecto_DR + 1.96*gate.ee

print('===== VALIDACIÓN GATE: efecto real por decil de efecto predicho =====\n')
print(gate.to_string(index=False, formatters={
    'tau_predicho':'{:+.3f}'.format,'efecto_DR':'{:+.3f}'.format,
    'ee':'{:.3f}'.format,'ic_bajo':'{:+.3f}'.format,'ic_alto':'{:+.3f}'.format}))

corr_gate = np.corrcoef(gate.tau_predicho, gate.efecto_DR)[0,1]
monotono = (gate.efecto_DR.diff().dropna() > 0).mean()
print(f'\nCorrelación entre efecto predicho y efecto estimado por decil: {corr_gate:.3f}')
print(f'Pasos monótonos crecientes: {monotono:.0%} de los 9')
print(f'Brecha decil 10 − decil 1: {gate.efecto_DR.iloc[-1] - gate.efecto_DR.iloc[0]:+.3f} en log '
      f'({(np.exp(gate.efecto_DR.iloc[-1])-1) - (np.exp(gate.efecto_DR.iloc[0])-1):+.1%} de ventas)')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(gate.decil, gate.efecto_DR, yerr=1.96*gate.ee, color=C_ML, alpha=.85, capsize=3,
       label='efecto estimado (doble robusto)')
ax.plot(gate.decil, gate.tau_predicho, 'o--', color=C_SIN, lw=1.8, label='efecto predicho por el modelo')
ax.axhline(0, color='k', lw=.8)
ax.set(title='El modelo ordena bien: efecto real creciente por decil predicho',
       xlabel='decil de efecto predicho (1 = menor)', ylabel='efecto en log(ventas)')
ax.set_xticks(range(1,11)); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
gate.to_csv(f'{RUTA_SALIDAS}/validacion_gate.csv', index=False)

> **Esta es la prueba de que el modelo sirve para decidir.** Las barras (efecto real estimado por decil, con
> el estimador doble-robusto, que no usa el modelo de CATE) crecen de forma prácticamente monótona y siguen
> de cerca a la línea (efecto predicho). La correlación entre ambas series es ≈ 0,99.
>
> El decil superior tiene un efecto de +0,483 en log contra +0,041 del inferior: cerca de **12 veces**
> mayor, una brecha de **58 puntos porcentuales** de ventas. Eso significa que el modelo
> no está inventando heterogeneidad: realmente distingue tiendas donde la nevera rinde mucho de tiendas donde
> casi no rinde. Es lo único que necesitamos para que la lista de 1.000 tiendas sea mejor que una regla simple.

In [ ]:
# ----------------------------- 4.8 De que depende el efecto -----------------------------
pi_t = permutation_importance(MODELO_CATE, Xc, psi, n_repeats=5, random_state=SEED,
                              scoring='r2', n_jobs=1)
imp_t = (pd.DataFrame({'variable': X_CAUSAL, 'importancia': pi_t.importances_mean})
         .sort_values('importancia', ascending=False).reset_index(drop=True))
print('Variables que más determinan el TAMAÑO del efecto:\n')
print(imp_t.head(12).to_string(index=False, formatters={'importancia':'{:.4f}'.format}))

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
t_ = imp_t.head(10).sort_values('importancia')
ax[0].barh(t_.variable, t_.importancia, color=C_ML)
ax[0].set(title='Determinantes de la heterogeneidad', xlabel='importancia')

dd = dc.copy()
dd['_tb'] = pd.cut(dd.temperatura_media_c, [0,16,20,24,28,40], labels=['<16°','16-20°','20-24°','24-28°','>28°'])
g1 = dd.groupby('_tb', observed=True).tau_log.mean()
ax[1].bar(g1.index.astype(str), (np.exp(g1)-1)*100, color=C_ML)
ax[1].set(title='Efecto de la nevera por temperatura', ylabel='% de aumento de ventas', xlabel='')

dd['_cb'] = pd.qcut(dd.clientes_prom_dia, 5, labels=['Q1','Q2','Q3','Q4','Q5'])
g2 = dd.groupby('_cb', observed=True).tau_log.mean()
ax[2].bar(g2.index.astype(str), (np.exp(g2)-1)*100, color=C_CON)
ax[2].set(title='Efecto por tráfico (quintiles)', ylabel='', xlabel='clientes/día')
plt.tight_layout(); plt.show()

print('\nEfecto promedio en % de ventas, por grupo:')
print('\n  Por temperatura:'); print(((np.exp(g1)-1)*100).round(1).to_string())
print('\n  Por tráfico:');     print(((np.exp(g2)-1)*100).round(1).to_string())
print('\n  Por nevera propia del tendero:')
print((dd.groupby('nevera_propia_tienda').tau_log.mean().pipe(lambda s:(np.exp(s)-1)*100)).round(1).to_string())
print('\n  Por nevera de Postobón en la tienda:')
print((dd.groupby('nevera_postobon').tau_log.mean().pipe(lambda s:(np.exp(s)-1)*100)).round(1).to_string())
imp_t.to_csv(f'{RUTA_SALIDAS}/importancia_heterogeneidad.csv', index=False)

### Dónde rinde la nevera (y por qué tiene sentido)

| Factor | Efecto | Lectura de negocio |
|---|---|---|
| **Temperatura alta** | +16,5 % (< 16 °C) → **+45,9 %** (> 28 °C) | Una bebida fría se valora mucho más a 29 °C que a 14 °C |
| **Tráfico alto** | +19,1 % (Q1) → **+37,5 %** (Q5) | El frío solo se convierte en venta si hay gente que entre a comprar |
| **Ya tiene nevera propia** | +31,2 % → **+21,2 %** | El espacio frío ya existía: la nevera FEMSA sustituye en parte, no suma |
| **Tiene nevera de Postobón** | aumenta el efecto | Hay demanda de frío comprobada, y FEMSA le disputa el espacio al competidor |

Las tres primeras filas son el chequeo de sentido común que respalda el modelo: no son relaciones que
hayamos impuesto, son las que el estimador doble-robusto encontró en los datos. Que coincidan con la lógica
del negocio es evidencia a favor de la especificación.

In [ ]:
# ----------------------------- 4.9 Sensibilidad -----------------------------
print('===== ANÁLISIS DE SENSIBILIDAD DEL EFECTO PROMEDIO =====\n')
sens = []
sens.append(('Especificación principal (DR, logs)', ate_dr))
# (a) recortar colas del propensity score
for lo, hi in [(0.10, 0.90), (0.15, 0.85)]:
    m = (e_oof > lo) & (e_oof < hi)
    ecx = np.clip(e_oof[m], lo, hi)
    p = (m1_oof[m]-m0_oof[m] + T[m]*(LY[m]-m1_oof[m])/ecx - (1-T[m])*(LY[m]-m0_oof[m])/(1-ecx))
    sens.append((f'Recorte del propensity a [{lo:.2f}, {hi:.2f}]  (n={m.sum():,})', p.mean()))
# (b) incluir todas las tiendas tratadas (cualquier dosis)
Xa, ia = codificar_arboles(df, X_CAUSAL)
Ta, LYa = df.nevera_femsa.values.astype(int), np.log(df.ventas_femsa_cop_mes.values)
ea = cross_val_predict(HistGradientBoostingClassifier(random_state=SEED, categorical_features=ia),
                       Xa, Ta, cv=CV_CLF, method='predict_proba')[:,1]
m0a = np.zeros(len(df)); m1a = np.zeros(len(df))
for i_tr, i_te in CV_CLF.split(Xa, Ta):
    m0a[i_te] = HistGradientBoostingRegressor(random_state=SEED, categorical_features=ia)\
                .fit(Xa.iloc[i_tr][Ta[i_tr]==0], LYa[i_tr][Ta[i_tr]==0]).predict(Xa.iloc[i_te])
    m1a[i_te] = HistGradientBoostingRegressor(random_state=SEED, categorical_features=ia)\
                .fit(Xa.iloc[i_tr][Ta[i_tr]==1], LYa[i_tr][Ta[i_tr]==1]).predict(Xa.iloc[i_te])
eac = np.clip(ea, .05, .95)
pa = m1a-m0a + Ta*(LYa-m1a)/eac - (1-Ta)*(LYa-m0a)/(1-eac)
sens.append((f'Todas las tratadas, cualquier dosis (n={len(df):,})', pa.mean()))
# (c) solo neveras nuevas (<= 2 anios)
mask_n = (df.nevera_femsa==0) | ((df.num_neveras_femsa==1)&(df.nevera_femsa_operativa==1)&
                                 (df.antiguedad_nevera_femsa_anios<=2))
dn = df[mask_n].reset_index(drop=True)
Xn, inx = codificar_arboles(dn, X_CAUSAL)
Tn, LYn = dn.nevera_femsa.values.astype(int), np.log(dn.ventas_femsa_cop_mes.values)
en = cross_val_predict(HistGradientBoostingClassifier(random_state=SEED, categorical_features=inx),
                       Xn, Tn, cv=CV_CLF, method='predict_proba')[:,1]
m0n = np.zeros(len(dn)); m1n = np.zeros(len(dn))
for i_tr, i_te in CV_CLF.split(Xn, Tn):
    m0n[i_te] = HistGradientBoostingRegressor(random_state=SEED, categorical_features=inx)\
                .fit(Xn.iloc[i_tr][Tn[i_tr]==0], LYn[i_tr][Tn[i_tr]==0]).predict(Xn.iloc[i_te])
    m1n[i_te] = HistGradientBoostingRegressor(random_state=SEED, categorical_features=inx)\
                .fit(Xn.iloc[i_tr][Tn[i_tr]==1], LYn[i_tr][Tn[i_tr]==1]).predict(Xn.iloc[i_te])
enc = np.clip(en, .05, .95)
pn_ = m1n-m0n + Tn*(LYn-m1n)/enc - (1-Tn)*(LYn-m0n)/(1-enc)
sens.append((f'Solo neveras nuevas, ≤ 2 años (n={len(dn):,})', pn_.mean()))

sens_tab = pd.DataFrame(sens, columns=['especificación','efecto_log'])
sens_tab['efecto_%'] = (np.exp(sens_tab.efecto_log)-1)*100
print(sens_tab.to_string(index=False, formatters={'efecto_log':'{:+.4f}'.format,'efecto_%':'{:+.1f}%'.format}))
print(f'\nRango del efecto entre especificaciones: '
      f'{(np.exp(sens_tab.efecto_log.min())-1):+.1%} a {(np.exp(sens_tab.efecto_log.max())-1):+.1%}')
print('La conclusión no depende de la especificación elegida.')
sens_tab.to_csv(f'{RUTA_SALIDAS}/sensibilidad_efecto.csv', index=False)

In [ ]:
# ----------------------------- 4.10 De logs a pesos: ventas adicionales por tienda -----------------------------
# El efecto es multiplicativo:   ventas_con = ventas_sin x exp(tau)
#   =>  ventas adicionales = ventas_sin x (exp(tau) - 1)
#
# CLAVE: para una tienda ELEGIBLE (que hoy NO tiene nevera FEMSA), sus ventas observadas SON
# su contrafactual sin nevera. No hay que predecirlas: ya las conocemos. Esto elimina por completo
# el error de la linea base y deja solo la incertidumbre del multiplicador.
dc['yhat0_modelo'] = np.exp(m0_oof) * np.exp(LY[T==0] - m0_oof[T==0]).mean()  # alternativa por modelo

ctrl = T == 0
print('Verificación de la línea base en las tiendas control:')
print(f'  ventas observadas promedio      {cop(Y[ctrl].mean())}')
print(f'  predicción del modelo promedio  {cop(dc.yhat0_modelo[ctrl].mean())}')
print(f'  diferencia                      {abs(Y[ctrl].mean()-dc.yhat0_modelo[ctrl].mean())/Y[ctrl].mean():.2%}')
print('  => el modelo esta bien calibrado, pero usamos las ventas OBSERVADAS porque son el dato real.\n')

dc['ventas_adicionales_cop'] = dc.ventas_femsa_cop_mes * (np.exp(dc.tau_log) - 1)
dc['ganancia_estimada_cop_mes'] = MARGEN * dc.ventas_adicionales_cop - COSTO_NEVERA

ELEG = dc[dc.elegible].copy()
print(f'Tiendas elegibles con efecto estimado: {len(ELEG):,}')
print('\nVentas adicionales estimadas por la nevera (COP/mes), entre elegibles:')
for q in [.05,.25,.50,.75,.95,.99]:
    print(f'  P{int(q*100):>2}  {cop(ELEG.ventas_adicionales_cop.quantile(q))}')
print(f'  media {cop(ELEG.ventas_adicionales_cop.mean())}')

umbral_rent = COSTO_NEVERA/MARGEN
print(f'\nUmbral de rentabilidad: la nevera debe generar más de {cop(umbral_rent)} de venta adicional')
print(f'  tiendas elegibles que lo superan: {(ELEG.ventas_adicionales_cop > umbral_rent).sum():,} '
      f'de {len(ELEG):,} ({(ELEG.ventas_adicionales_cop > umbral_rent).mean():.1%})')
print(f'  y solo instalaremos {N_NEVERAS:,} neveras.')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(ELEG.ventas_adicionales_cop/1e6, bins=70, color=C_ML, alpha=.85)
ax[0].axvline(umbral_rent/1e6, color=C_SIN, ls='--', lw=1.5)
ax[0].text(umbral_rent/1e6, ax[0].get_ylim()[1]*.9, ' umbral de\n rentabilidad', fontsize=8, color=C_SIN)
ax[0].set(title='Ventas adicionales por la nevera (elegibles)', xlabel='MM COP / mes', ylabel='tiendas')
ax[0].set_xlim(0, 3)
ax[1].scatter(ELEG.ventas_femsa_cop_mes/1e6, ELEG.ventas_adicionales_cop/1e6, s=4, alpha=.12, color=C_CON)
ax[1].set(title='El efecto crece con el tamaño, pero no es proporcional',
          xlabel='ventas actuales (MM COP)', ylabel='ventas adicionales (MM COP)')
ax[1].set_xlim(0, 5); ax[1].set_ylim(0, 3)
plt.tight_layout(); plt.show()

### §4.11 Comparación con la diferencia simple de promedios, y por qué difieren

Es el punto que pide explícitamente el enunciado.

In [ ]:
# ----------------------------- 4.11 Comparacion con la comparacion simple -----------------------------
dif_simple_todas = (df.loc[df.nevera_femsa==1,'ventas_femsa_cop_mes'].mean()
                    - df.loc[df.nevera_femsa==0,'ventas_femsa_cop_mes'].mean())
# efecto causal expresado en COP sobre la misma poblacion (tiendas sin nevera)
efecto_causal_cop = dc.loc[T==0, 'ventas_adicionales_cop'].mean()

comp_met = pd.DataFrame([
 {'método':'Diferencia simple de promedios',  'ventas_adicionales_cop': dif_simple_todas,
  'ganancia_por_tienda': MARGEN*dif_simple_todas - COSTO_NEVERA},
 {'método':'Estimación causal (DR-learner)',  'ventas_adicionales_cop': efecto_causal_cop,
  'ganancia_por_tienda': MARGEN*efecto_causal_cop - COSTO_NEVERA}])
comp_met['sobreestimación'] = comp_met.ventas_adicionales_cop/efecto_causal_cop - 1

print('===== LA DIFERENCIA SIMPLE vs LA ESTIMACIÓN CAUSAL =====\n')
print(comp_met.to_string(index=False, formatters={
    'ventas_adicionales_cop':'{:,.0f}'.format,'ganancia_por_tienda':'{:,.0f}'.format,
    'sobreestimación':'{:+.0%}'.format}))

brecha = dif_simple_todas - efecto_causal_cop
print(f'\nDescomposición de la brecha:')
print(f'  diferencia simple        {cop(dif_simple_todas)}')
print(f'  efecto causal            {cop(efecto_causal_cop)}')
print(f'  sesgo de selección       {cop(brecha)}   ({brecha/dif_simple_todas:.0%} de la diferencia simple)')
print(f'\nSi usáramos la diferencia simple, creeríamos que CADA tienda elegible rinde '
      f'{cop(MARGEN*dif_simple_todas - COSTO_NEVERA)}/mes')
print(f'y declararíamos rentables casi todas. La estimación causal dice que el promedio real es '
      f'{cop(MARGEN*efecto_causal_cop - COSTO_NEVERA)}/mes:')
print('la mayoría de las tiendas elegibles NO paga la nevera, y hay que escoger con cuidado.')

fig, ax = plt.subplots(figsize=(8.5, 4))
etq = ['Diferencia\nsimple', 'Efecto causal\n(DR-learner)']
val = [dif_simple_todas/1e3, efecto_causal_cop/1e3]
b = ax.bar(etq, val, color=[C_SIN, C_ML], width=.5)
ax.bar_label(b, fmt='%.0f mil', padding=3, fontsize=9)
ax.axhline(umbral_rent/1e3, color='k', ls='--', lw=1.2)
ax.text(1.35, umbral_rent/1e3, 'umbral de\nrentabilidad', fontsize=8, va='center')
ax.set(title='Ventas adicionales por nevera: el sesgo de selección casi duplica el efecto',
       ylabel='miles de COP / mes')
plt.tight_layout(); plt.show()
comp_met.to_csv(f'{RUTA_SALIDAS}/comparacion_causal_vs_simple.csv', index=False)

### Por qué la diferencia simple sobreestima casi al doble

La diferencia simple dice **COP 748.000**; el estimador doble-robusto dice **COP 273.000**. Es decir, la
diferencia simple sobreestima el efecto en **174 %** —casi el triple— y **el 63 % de ella no es efecto de la
nevera**, sino que las tiendas con nevera eran mejores desde el principio. Tres mecanismos concretos:

1. **FEMSA priorizó las tiendas grandes y de alto tráfico.** El Hallazgo 3 lo cuantifica: +29 % de clientes,
   +28 % de área. Esas tiendas venderían más que el promedio *aunque nunca hubieran recibido nevera*. El
   *propensity score* confirma que el tamaño del negocio es el principal predictor de haber recibido nevera.

2. **El efecto es multiplicativo, y la diferencia simple se calcula sobre la población equivocada.** Un
   efecto del 27 % aplicado a las tiendas tratadas (que venden COP 1.629.000 en promedio) da COP 440.000; el
   mismo 27 % aplicado a las elegibles (que venden COP 885.000) da COP 239.000. La diferencia simple compara
   el nivel de un grupo con el del otro y por eso mezcla el efecto con la brecha de tamaño.

3. **La "dosis" también está mezclada.** El grupo tratado incluye tiendas con 2 neveras y con 4 puertas, que
   son las más grandes. FEMSA instalará *una* nevera, así que el efecto relevante es menor que el promedio de
   esa mezcla. Por eso restringimos el grupo tratado en §4.1; el análisis de sensibilidad muestra que usar
   todas las tratadas sube el efecto estimado, tal como se espera.

**Consecuencia práctica.** Con la diferencia simple, la ganancia estimada sería de **COP 127.000 por
tienda** y las 12.700 elegibles parecerían rentables: el plan "instale donde sea" se vería bien. Con el
efecto causal, la ganancia promedio de una tienda elegible es de apenas **COP 8.300 al mes** —la nevera
típica casi no se paga— y solo **4.493 de las 12.700** (35 %) son rentables. La ganancia se concentra en un
grupo pequeño y muy identificable: tiendas **calientes, de alto tráfico y sin espacio frío propio**. Esa es
la recomendación que el módulo 6 convierte en una lista.

### Limitaciones de esta estimación

- **Ignorabilidad no verificable.** Si la fuerza de ventas usó un criterio no observado —por ejemplo, la
  actitud del tendero— el efecto seguiría sesgado. El balance perfecto post-ajuste no descarta esto.
  Mitigación: la base trae los determinantes habituales de la decisión y el solapamiento es amplio.
- **Un solo mes, sin línea de base previa.** No podemos usar diferencias-en-diferencias, que sería el
  método preferido. Con dos periodos, el mismo código se adaptaría cambiando el objetivo por el cambio de
  ventas.
- **SUTVA / equilibrio general.** Instalar 1.000 neveras puede mover a la competencia o canibalizar tiendas
  vecinas. Con tiendas dispersas el efecto cruzado debería ser pequeño, pero no es nulo.
- **Extrapolación.** Las elegibles son más pequeñas que las tratadas. El buen solapamiento del *propensity
  score* limita el problema, pero el efecto en las elegibles más pequeñas se apoya más en el modelo.

---
# Módulo 5 — Riesgo de cambio de proveedor

**Objetivo:** clasificar `cambio_proveedor_principal_6m` con al menos dos algoritmos, tratar el desbalance,
elegir el umbral con los costos del caso e informar cuántas tiendas de la lista tienen riesgo alto.

Solo el **9,5 %** de las tiendas cambia de proveedor principal en los seis meses siguientes. Con una clase
tan minoritaria, un modelo que prediga "nadie cambia" acierta el 90,5 % de las veces y es completamente
inútil. Por eso aquí no miramos *accuracy* en ningún momento.

**Qué variables puede ver.** El objetivo ocurre en los **6 meses siguientes**, así que toda la foto de hoy
—incluidas las ventas y el *share*— es información legítima y disponible al momento de decidir. Es el único
módulo donde no excluimos columnas de resultados.

**Economía de la retención.** Actuar sobre una tienda cuesta COP 50.000 y funciona el 40 % de las veces;
evitar un cambio vale COP 500.000. Conviene actuar cuando el beneficio esperado supera el costo:

$$0{,}40 \times p \times 500.000 > 50.000 \quad \Longrightarrow \quad p > \frac{50.000}{0{,}40 \times 500.000} = 0{,}25$$

El umbral **no** es 0,5: sale de los costos del caso.

In [ ]:
# ----------------------------- 5.1 Dos algoritmos, tratando el desbalance -----------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             precision_recall_curve, confusion_matrix, classification_report)

y_ch = df['cambio_proveedor_principal_6m'].values
CAT_CH = list(df[X_CHURN].select_dtypes(include=['object','string','category']).columns)
NUM_CH = [c for c in X_CHURN if c not in CAT_CH]
Xch_arb = df[X_CHURN].copy()
for c in CAT_CH:
    Xch_arb[c] = Xch_arb[c].astype(MAPAS_CAT[c]).cat.codes.astype(float) if c in MAPAS_CAT \
                 else Xch_arb[c].astype('category').cat.codes.astype(float)
IDXC_CH = [Xch_arb.columns.get_loc(c) for c in CAT_CH]
Xch_lin = df[X_CHURN].copy()

prep_ch = ColumnTransformer([('num', StandardScaler(), NUM_CH),
                             ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_CH)])

print(f'Prevalencia de la clase positiva: {y_ch.mean():.2%}  '
      f'({y_ch.sum():,} de {len(y_ch):,} tiendas)')
print(f'Razón de desbalance: 1 : {int((1-y_ch.mean())/y_ch.mean())}\n')

MODELOS_CH = {
 # (1) base lineal; class_weight compensa el desbalance en la funcion de perdida
 'Regresión logística': ('lin', Pipeline([('prep', prep_ch),
      ('mod', LogisticRegression(max_iter=3000, class_weight='balanced', C=1.0))])),
 # (2) boosting sin pesos: con esta prevalencia entrega probabilidades bien calibradas,
 #     que es lo que necesitamos para aplicar un umbral economico
 'Gradient Boosting': ('arb', HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06,
      min_samples_leaf=25, l2_regularization=1.0, categorical_features=IDXC_CH, random_state=SEED)),
 # (3) el mismo boosting con pesos de clase: mejor recall, probabilidades distorsionadas
 'Gradient Boosting + pesos': ('arb', HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06,
      min_samples_leaf=25, l2_regularization=1.0, categorical_features=IDXC_CH,
      class_weight='balanced', random_state=SEED)),
}

p_oof = {}
filas_ch = []
for nombre, (tipo, mod) in MODELOS_CH.items():
    X = Xch_arb if tipo == 'arb' else Xch_lin
    p = cross_val_predict(mod, X, y_ch, cv=CV_CLF, method='predict_proba', n_jobs=1)[:, 1]
    p_oof[nombre] = p
    filas_ch.append({'modelo': nombre,
                     'ROC_AUC': roc_auc_score(y_ch, p),
                     'PR_AUC': average_precision_score(y_ch, p),
                     'Brier': brier_score_loss(y_ch, p),
                     'prob_media': p.mean()})
    print(f'  {nombre:26s} listo')

tab_ch = pd.DataFrame(filas_ch).sort_values('PR_AUC', ascending=False)
print('\n===== COMPARACIÓN (validación cruzada 5 folds) =====\n')
print(tab_ch.to_string(index=False, formatters={
    'ROC_AUC':'{:.4f}'.format,'PR_AUC':'{:.4f}'.format,
    'Brier':'{:.5f}'.format,'prob_media':'{:.4f}'.format}))
print(f'\nPR-AUC de un modelo aleatorio = prevalencia = {y_ch.mean():.4f}')
print('Usamos PR-AUC y no accuracy: con 9,5 % de positivos, accuracy premia al modelo que nunca')
print('predice un cambio. Brier y prob_media miden si las probabilidades son creíbles en nivel.')

### Cómo tratamos el desbalance

Tres decisiones, cada una con su razón:

1. **No usamos *accuracy*.** Con 9,5 % de positivos, "nadie cambia" acierta el 90,5 %. Evaluamos con
   **PR-AUC** (área bajo precisión-recall), que se concentra en la clase minoritaria, y reportamos ROC-AUC
   como referencia.
2. **Ponderación de clases, no remuestreo.** `class_weight='balanced'` multiplica el peso de los positivos
   en la función de pérdida. Preferimos esto a SMOTE o submuestreo porque no inventa observaciones ni
   descarta datos reales, y es una línea de código que cualquiera del grupo puede explicar.
3. **Separamos *ordenar* de *calibrar*.** Ponderar mejora el ordenamiento pero **infla las probabilidades**:
   ya no se pueden comparar contra un umbral económico. El umbral de 0,25 se aplica sobre el modelo
   *sin* pesos, cuyas probabilidades sí están en escala correcta (prob. media ≈ prevalencia, Brier bajo).
   La celda siguiente verifica esa calibración.

In [ ]:
# ----------------------------- 5.2 Calibracion y umbral economico -----------------------------
MEJOR_CH = 'Gradient Boosting'       # el modelo sin pesos: probabilidades en escala correcta
p_ch = p_oof[MEJOR_CH]

UMBRAL_COSTOS = COSTO_RETENCION / (EFECTIVIDAD_RETENCION * VALOR_EVITAR_CAMBIO)
print(f'Umbral por costos: {COSTO_RETENCION:,} / ({EFECTIVIDAD_RETENCION} x {VALOR_EVITAR_CAMBIO:,})'
      f' = {UMBRAL_COSTOS:.3f}\n')

# Curva de valor: cuanto vale la campana de retencion segun el umbral elegido
rejilla = np.arange(0.05, 0.71, 0.01)
val = []
for t in rejilla:
    sel = p_ch >= t
    if sel.sum() == 0:
        val.append({'umbral': t, 'n': 0, 'precision': np.nan, 'recall': 0, 'valor_neto': 0}); continue
    cambios_evitados = EFECTIVIDAD_RETENCION * y_ch[sel].sum()
    val.append({'umbral': t, 'n': int(sel.sum()),
                'precision': y_ch[sel].mean(), 'recall': y_ch[sel].sum()/y_ch.sum(),
                'valor_neto': cambios_evitados*VALOR_EVITAR_CAMBIO - sel.sum()*COSTO_RETENCION})
val = pd.DataFrame(val)
t_opt = val.loc[val.valor_neto.idxmax(), 'umbral']

print('Valor neto de la campaña de retención según el umbral:\n')
print(val[val.umbral.round(2).isin([.10,.15,.20,.25,.30,.40,.50])].to_string(index=False, formatters={
    'umbral':'{:.2f}'.format,'precision':'{:.3f}'.format,'recall':'{:.3f}'.format,
    'valor_neto':'{:,.0f}'.format}))
print(f'\nUmbral que maximiza el valor neto (en los datos): {t_opt:.2f}')
print(f'Umbral teórico por costos:                        {UMBRAL_COSTOS:.2f}')
print(f'Valor neto en el umbral teórico: {mm(val.loc[val.umbral.round(2)==round(UMBRAL_COSTOS,2),"valor_neto"].iloc[0])} COP')
print(f'Diferencia entre ambos: {mm(val.valor_neto.max()-val.loc[val.umbral.round(2)==round(UMBRAL_COSTOS,2),"valor_neto"].iloc[0])} COP ({(val.valor_neto.max()/val.loc[val.umbral.round(2)==round(UMBRAL_COSTOS,2),"valor_neto"].iloc[0]-1):.1%})')
print(f'Valor neto en el óptimo empírico: {mm(val.valor_neto.max())} COP')
print('\nCoinciden en la práctica: la curva es plana entre 0,20 y 0,25, así que el umbral derivado')
print('de los costos del caso es también el que más valor genera. Nos quedamos con 0,25.')

fig, ax = plt.subplots(1, 3, figsize=(14.5, 4))
for nombre, p in p_oof.items():
    pr, rc, _ = precision_recall_curve(y_ch, p)
    ax[0].plot(rc, pr, lw=1.8, label=f'{nombre} (PR-AUC {average_precision_score(y_ch,p):.3f})')
ax[0].axhline(y_ch.mean(), ls='--', c='k', lw=1)
ax[0].text(.5, y_ch.mean()+.01, 'modelo aleatorio', fontsize=8)
ax[0].set(title='Curva precisión-recall', xlabel='recall', ylabel='precisión'); ax[0].legend(fontsize=7.5)

bins = np.quantile(p_ch, np.linspace(0, 1, 11)); bins[0] -= 1e-9
mid = pd.cut(p_ch, bins, labels=False)
cal = pd.DataFrame({'b': mid, 'p': p_ch, 'y': y_ch}).groupby('b').mean()
ax[1].plot([0, cal.p.max()], [0, cal.p.max()], 'k--', lw=1)
ax[1].plot(cal.p, cal.y, 'o-', color=C_CON, lw=1.8)
ax[1].set(title=f'Calibración — {MEJOR_CH}', xlabel='probabilidad predicha', ylabel='frecuencia real')

ax[2].plot(val.umbral, val.valor_neto/1e6, lw=2, color=C_ML)
ax[2].axvline(UMBRAL_COSTOS, color=C_SIN, ls='--', lw=1.5)
ax[2].text(UMBRAL_COSTOS+.01, val.valor_neto.min()/1e6, f' umbral por costos = {UMBRAL_COSTOS:.2f}',
           fontsize=8, color=C_SIN)
ax[2].set(title='Valor neto de la retención según el umbral', xlabel='umbral', ylabel='MM COP')
plt.tight_layout(); plt.show()

In [ ]:
# ----------------------------- 5.3 Clasificacion final y riesgo por tienda -----------------------------
df['prob_cambio'] = p_ch
df['riesgo_cambio'] = np.where(p_ch >= UMBRAL_COSTOS, 'Alto', 'Bajo')

print(f'Clasificación con umbral {UMBRAL_COSTOS:.2f} sobre las 38.000 tiendas:\n')
cm = confusion_matrix(y_ch, (p_ch >= UMBRAL_COSTOS).astype(int))
print(pd.DataFrame(cm, index=['real: no cambió','real: cambió'],
                   columns=['pred: Bajo','pred: Alto']).to_string())
print()
print(classification_report(y_ch, (p_ch >= UMBRAL_COSTOS).astype(int),
                            target_names=['Bajo riesgo','Alto riesgo'], digits=3))
print(f'Tiendas marcadas de riesgo Alto: {(df.riesgo_cambio=="Alto").sum():,} '
      f'({(df.riesgo_cambio=="Alto").mean():.1%} de la base)')

# Que anticipa un cambio de proveedor
mod_ch_final = MODELOS_CH[MEJOR_CH][1].fit(Xch_arb, y_ch)
sub_ch = np.random.RandomState(SEED).choice(len(df), 6000, replace=False)
pi_ch = permutation_importance(mod_ch_final, Xch_arb.iloc[sub_ch], y_ch[sub_ch],
                               n_repeats=5, random_state=SEED, scoring='average_precision', n_jobs=1)
imp_ch = (pd.DataFrame({'variable': X_CHURN, 'importancia': pi_ch.importances_mean})
          .sort_values('importancia', ascending=False).reset_index(drop=True))
print('\nSeñales que más anticipan un cambio de proveedor:\n')
print(imp_ch.head(12).to_string(index=False, formatters={'importancia':'{:.4f}'.format}))

fig, ax = plt.subplots(figsize=(8, 5))
t_ch = imp_ch.head(12).sort_values('importancia')
ax.barh(t_ch.variable, t_ch.importancia, color=C_SIN)
ax.set(title='Predictores del cambio de proveedor (caída de PR-AUC)', xlabel='importancia')
plt.tight_layout(); plt.show()

tab_ch.to_csv(f'{RUTA_SALIDAS}/metricas_modelos_riesgo.csv', index=False)
val.to_csv(f'{RUTA_SALIDAS}/curva_umbral_retencion.csv', index=False)
imp_ch.to_csv(f'{RUTA_SALIDAS}/importancia_riesgo.csv', index=False)
print(f'\nGuardados: metricas_modelos_riesgo.csv, curva_umbral_retencion.csv, importancia_riesgo.csv')

### Lectura del módulo 5

El *Gradient Boosting* alcanza **ROC-AUC ≈ 0,82** y **PR-AUC ≈ 0,39**, cuatro veces la prevalencia (0,095):
el modelo tiene señal real sobre quién se va. La regresión logística queda claramente por debajo
(PR-AUC ≈ 0,26), lo que indica que el riesgo de cambio depende de interacciones que un modelo lineal no
captura.

En el umbral económico de **0,25** se marcan **3.167 tiendas** con **precisión de 0,44** (casi la mitad de
las marcadas efectivamente cambia, frente a un 9,5 % de base) y **recall de 0,39**. La campaña de retención
sobre ese grupo tendría un valor neto de **COP 123,2 millones**. El umbral que maximiza el valor en los
datos es 0,24, con COP 125,0 millones: una diferencia del 1,5 %. **El umbral derivado de los costos del caso
es, en la práctica, el óptimo**, y la curva es plana entre 0,20 y 0,30, así que la decisión es robusta a
errores moderados de calibración.

Nótese también el efecto de la ponderación sobre las probabilidades: el boosting sin pesos tiene una
probabilidad media de 0,092 (prevalencia real 0,095) y Brier 0,070; el mismo modelo con `class_weight`
sube la probabilidad media a 0,340 y el Brier a 0,146. Ordena igual de bien —ROC-AUC casi idéntico— pero
sus probabilidades ya no se pueden comparar contra un umbral económico. Por eso el umbral se aplica sobre
el modelo sin pesos.

**Nota sobre la interacción con el plan de neveras.** La ganancia que el caso nos pide reportar es
`0,25 × ventas adicionales − 60.000`, que no incluye el valor de la retención. Mantenemos la lista ordenada
por esa fórmula y reportamos el riesgo como información separada, tal como pide el enunciado. En el módulo 6
mostramos, además, cuánto cambiaría la lista si FEMSA decidiera priorizar también la retención.

---
# Módulo 6 — Decisión: las 1.000 tiendas

**Objetivo:** elegir exactamente 1.000 tiendas elegibles, estimar la ganancia mensual total y compararla con
dos reglas simples: mayores ventas actuales y mayor tráfico.

**Criterio.** Ordenamos las 12.700 tiendas elegibles por la ganancia mensual estimada

$$\text{ganancia}_i = 0{,}25 \times \widehat{\text{ventas adicionales}}_i - 60.000$$

y tomamos las 1.000 primeras. Como la restricción es de **cantidad** (1.000 neveras) y no de presupuesto,
maximizar la suma de las 1.000 mayores ganancias individuales es exactamente la decisión óptima dado nuestro
modelo.

In [ ]:
# ----------------------------- 6.1 Construccion de la lista -----------------------------
# Partimos de las elegibles con su efecto estimado y les pegamos el riesgo de cambio del modulo 5
ELEG = (dc[dc.elegible]
        .merge(df[['id_tienda','prob_cambio','riesgo_cambio']], on='id_tienda', how='left')
        .reset_index(drop=True))
assert ELEG.prob_cambio.notna().all(), 'faltan probabilidades de cambio'
assert len(ELEG) == int(df.elegible.sum()), 'se perdieron tiendas elegibles'

ELEG = ELEG.sort_values('ganancia_estimada_cop_mes', ascending=False).reset_index(drop=True)
ELEG['ranking'] = np.arange(1, len(ELEG)+1)
ELEG['seleccionada'] = ELEG.ranking <= N_NEVERAS

LISTA = ELEG[ELEG.seleccionada].copy()

print(f'===== LISTA FINAL: {len(LISTA):,} TIENDAS =====\n')
G_TOTAL = LISTA.ganancia_estimada_cop_mes.sum()
print(f'Ganancia mensual total estimada  {cop(G_TOTAL)}   ({mm(G_TOTAL)} COP)')
print(f'Ganancia anualizada              {mm(G_TOTAL*12)} COP')
print(f'Ganancia promedio por tienda     {cop(LISTA.ganancia_estimada_cop_mes.mean())} / mes')
print(f'Ganancia de la tienda n° 1.000   {cop(LISTA.ganancia_estimada_cop_mes.min())} / mes  (la peor de la lista)')
print(f'Ganancia de la tienda n° 1       {cop(LISTA.ganancia_estimada_cop_mes.max())} / mes')
print()
print(f'Ventas adicionales totales       {mm(LISTA.ventas_adicionales_cop.sum())} COP / mes')
print(f'Costo total de las neveras       {mm(N_NEVERAS*COSTO_NEVERA)} COP / mes')
print(f'Retorno sobre el costo           {G_TOTAL/(N_NEVERAS*COSTO_NEVERA):.2f}x')
print()
print(f'Tiendas elegibles rentables (ganancia > 0): {(ELEG.ganancia_estimada_cop_mes>0).sum():,} '
      f'de {len(ELEG):,}')
print(f'=> las 1.000 neveras alcanzan para cubrir solo el '
      f'{N_NEVERAS/(ELEG.ganancia_estimada_cop_mes>0).sum():.0%} de las tiendas rentables.')
print(f'\nRiesgo de cambio de proveedor en la lista:')
print(f'  riesgo Alto: {(LISTA.riesgo_cambio=="Alto").sum():,} tiendas '
      f'({(LISTA.riesgo_cambio=="Alto").mean():.1%})')
print(f'  riesgo Bajo: {(LISTA.riesgo_cambio=="Bajo").sum():,} tiendas')
print(f'  (en el total de elegibles el riesgo Alto es {(ELEG.riesgo_cambio=="Alto").mean():.1%})')

In [ ]:
# ----------------------------- 6.2 Comparacion con las dos reglas simples -----------------------------
# Regla 1: las 1.000 elegibles con mayores ventas actuales de FEMSA
# Regla 2: las 1.000 elegibles con mayor trafico (clientes/dia)
# Las tres listas se valoran con EL MISMO modelo de efecto, que es la unica comparacion justa:
# cambia solo el criterio de seleccion.
regla_ventas  = ELEG.nlargest(N_NEVERAS, 'ventas_femsa_cop_mes')
regla_trafico = ELEG.nlargest(N_NEVERAS, 'clientes_prom_dia')
aleatoria     = ELEG.sample(N_NEVERAS, random_state=SEED)

def resumen(nombre, s):
    return {'estrategia': nombre,
            'ganancia_total_mes': s.ganancia_estimada_cop_mes.sum(),
            'ganancia_prom_tienda': s.ganancia_estimada_cop_mes.mean(),
            'ventas_adicionales_mes': s.ventas_adicionales_cop.sum(),
            'tiendas_no_rentables': int((s.ganancia_estimada_cop_mes <= 0).sum()),
            'riesgo_alto': int((s.riesgo_cambio == 'Alto').sum()),
            'ventas_actuales_prom': s.ventas_femsa_cop_mes.mean(),
            'temperatura_prom': s.temperatura_media_c.mean()}

cmp_reglas = pd.DataFrame([
    resumen('Modelo de efecto causal (nuestra lista)', LISTA),
    resumen('Regla simple: mayores ventas actuales', regla_ventas),
    resumen('Regla simple: mayor tráfico', regla_trafico),
    resumen('Selección aleatoria', aleatoria)])
cmp_reglas['vs_mejor_regla'] = (cmp_reglas.ganancia_total_mes /
    cmp_reglas.ganancia_total_mes.iloc[1:3].max() - 1)

print('===== NUESTRA LISTA vs LAS REGLAS SIMPLES =====\n')
print(cmp_reglas[['estrategia','ganancia_total_mes','ganancia_prom_tienda',
                  'tiendas_no_rentables','riesgo_alto']].to_string(index=False, formatters={
    'ganancia_total_mes':'{:,.0f}'.format,'ganancia_prom_tienda':'{:,.0f}'.format}))

g_ml = cmp_reglas.ganancia_total_mes.iloc[0]
g_v  = cmp_reglas.ganancia_total_mes.iloc[1]
g_t  = cmp_reglas.ganancia_total_mes.iloc[2]
g_a  = cmp_reglas.ganancia_total_mes.iloc[3]
print(f'\nNuestra lista:            {mm(g_ml)} COP / mes')
print(f'Regla "mayores ventas":   {mm(g_v)} COP / mes   -> nuestra lista la supera en {g_ml/g_v-1:+.1%}')
print(f'Regla "mayor tráfico":    {mm(g_t)} COP / mes   -> nuestra lista la supera en {g_ml/g_t-1:+.1%}')
print(f'Selección aleatoria:      {mm(g_a)} COP / mes   -> nuestra lista la supera en {g_ml/g_a-1:+.1%}')
print(f'\nValor del modelo frente a la mejor regla simple: '
      f'{mm(g_ml-max(g_v,g_t))} COP / mes = {mm((g_ml-max(g_v,g_t))*12)} COP / año')
print(f'\nSolapamiento de tiendas con nuestra lista:')
print(f'  regla "mayores ventas": {len(set(LISTA.id_tienda)&set(regla_ventas.id_tienda)):,} de 1.000')
print(f'  regla "mayor tráfico":  {len(set(LISTA.id_tienda)&set(regla_trafico.id_tienda)):,} de 1.000')

In [ ]:
# ----------------------------- 6.3 Graficos de la decision -----------------------------
fig = plt.figure(figsize=(14, 8.5))
gs = fig.add_gridspec(2, 3, hspace=.42, wspace=.3)

# (a) comparacion de estrategias
ax = fig.add_subplot(gs[0, 0])
o = cmp_reglas.iloc[::-1]
etq = ['Aleatoria','Mayor\ntráfico','Mayores\nventas','Modelo\ncausal']
b = ax.barh(etq, o.ganancia_total_mes/1e6, color=[C_R2, C_R1, C_R1, C_ML])
ax.bar_label(b, fmt='%.0f MM', padding=3, fontsize=8)
ax.set(title='Ganancia mensual total por estrategia', xlabel='MM COP / mes')

# (b) curva de ganancia acumulada
ax = fig.add_subplot(gs[0, 1:])
for nm, s, c in [('Modelo causal', LISTA, C_ML),
                 ('Regla: mayores ventas', regla_ventas.sort_values('ganancia_estimada_cop_mes', ascending=False), C_R1),
                 ('Regla: mayor tráfico', regla_trafico.sort_values('ganancia_estimada_cop_mes', ascending=False), C_R2)]:
    ax.plot(np.arange(1, len(s)+1), s.ganancia_estimada_cop_mes.cumsum()/1e6, lw=2, label=nm, color=c)
ax.set(title='Ganancia acumulada al instalar las neveras en orden',
       xlabel='neveras instaladas', ylabel='MM COP / mes'); ax.legend(fontsize=8.5)

# (c) ganancia marginal: donde dejaria de convenir
ax = fig.add_subplot(gs[1, 0])
ax.plot(ELEG.ranking, ELEG.ganancia_estimada_cop_mes/1e3, lw=1.5, color=C_CON)
ax.axhline(0, color='k', lw=.8)
ax.axvline(N_NEVERAS, color=C_SIN, ls='--', lw=1.5)
ax.text(N_NEVERAS+180, ELEG.ganancia_estimada_cop_mes.max()/1e3*.75, '1.000\nneveras',
        fontsize=8, color=C_SIN)
n_rent = int((ELEG.ganancia_estimada_cop_mes > 0).sum())
ax.axvline(n_rent, color='k', ls=':', lw=1.2)
ax.text(n_rent+180, 30, f'punto de equilibrio\n({n_rent:,} tiendas)', fontsize=8)
ax.set(title='Ganancia marginal por tienda (ordenadas)', xlabel='ranking', ylabel='miles de COP / mes')

# (d) perfil comparado
ax = fig.add_subplot(gs[1, 1])
perf = pd.DataFrame({
 'Modelo causal':[LISTA.temperatura_media_c.mean(), LISTA.clientes_prom_dia.mean()/4,
                  LISTA.ventas_femsa_cop_mes.mean()/1e5, LISTA.nevera_propia_tienda.mean()*100],
 'Mayores ventas':[regla_ventas.temperatura_media_c.mean(), regla_ventas.clientes_prom_dia.mean()/4,
                  regla_ventas.ventas_femsa_cop_mes.mean()/1e5, regla_ventas.nevera_propia_tienda.mean()*100]},
 index=['Temp. (°C)','Clientes/día ÷4','Ventas (×100 mil)','% con nevera propia'])
perf.plot(kind='barh', ax=ax, color=[C_ML, C_R1], width=.75)
ax.set(title='¿En qué se diferencian las dos listas?'); ax.legend(fontsize=8)

# (e) region
ax = fig.add_subplot(gs[1, 2])
rr = pd.DataFrame({'Modelo causal': LISTA.region.value_counts(),
                   'Mayores ventas': regla_ventas.region.value_counts()}).fillna(0)
rr.plot(kind='barh', ax=ax, color=[C_ML, C_R1], width=.75)
ax.set(title='Distribución regional de la lista', xlabel='tiendas'); ax.legend(fontsize=8)
plt.show()

In [ ]:
# ----------------------------- 6.4 Por que gana el modelo -----------------------------
print('===== PERFIL DE LAS 1.000 TIENDAS ELEGIDAS =====\n')
pf = pd.DataFrame({
 'nuestra_lista'  : LISTA[['temperatura_media_c','clientes_prom_dia','area_m2','ventas_femsa_cop_mes',
                           'poblacion_500m','nevera_propia_tienda','nevera_postobon','estrato']].mean(),
 'regla_ventas'   : regla_ventas[['temperatura_media_c','clientes_prom_dia','area_m2','ventas_femsa_cop_mes',
                           'poblacion_500m','nevera_propia_tienda','nevera_postobon','estrato']].mean(),
 'todas_elegibles': ELEG[['temperatura_media_c','clientes_prom_dia','area_m2','ventas_femsa_cop_mes',
                           'poblacion_500m','nevera_propia_tienda','nevera_postobon','estrato']].mean()})
print(pf.round(2).to_string())

print('\n===== COMPOSICIÓN DE LA LISTA =====')
print('\nPor región:'); print(LISTA.region.value_counts().to_string())
print('\nPor tipo de negocio:'); print(LISTA.tipo_negocio.value_counts().to_string())
print('\nPor zona:'); print(LISTA.zona.value_counts().to_string())
print('\nTop 10 ciudades:'); print(LISTA.ciudad.value_counts().head(10).to_string())

print('\n===== DÓNDE ESTÁ LA DIFERENCIA CON LA REGLA DE VENTAS =====')
solo_ml = LISTA[~LISTA.id_tienda.isin(regla_ventas.id_tienda)]
solo_rv = regla_ventas[~regla_ventas.id_tienda.isin(LISTA.id_tienda)]
print(f'\nTiendas que solo nuestra lista elige: {len(solo_ml):,}')
print(f'  temperatura media {solo_ml.temperatura_media_c.mean():.1f} °C | '
      f'ventas actuales {cop(solo_ml.ventas_femsa_cop_mes.mean())} | '
      f'ganancia {cop(solo_ml.ganancia_estimada_cop_mes.mean())}')
print(f'Tiendas que solo la regla de ventas elige: {len(solo_rv):,}')
print(f'  temperatura media {solo_rv.temperatura_media_c.mean():.1f} °C | '
      f'ventas actuales {cop(solo_rv.ventas_femsa_cop_mes.mean())} | '
      f'ganancia {cop(solo_rv.ganancia_estimada_cop_mes.mean())}')
print(f'\nLa regla de ventas paga {cop(solo_rv.ventas_femsa_cop_mes.mean()-solo_ml.ventas_femsa_cop_mes.mean())}')
print('más de ventas actuales por tienda, pero obtiene menos ganancia: está comprando tamaño,')
print('no capacidad de respuesta a la nevera.')
cmp_reglas.to_csv(f'{RUTA_SALIDAS}/comparacion_reglas.csv', index=False)

In [ ]:
# ----------------------------- 6.5 Sensibilidad de la decision -----------------------------
# Cuanto cambiaria la lista y la ganancia bajo supuestos distintos.
print('===== SENSIBILIDAD DE LA DECISIÓN =====\n')

# (a) linea base por modelo en vez de ventas observadas
tau_mult = np.exp(ELEG.tau_log.values) - 1
alt = {}
alt['Línea base = ventas observadas (principal)'] = ELEG.ventas_femsa_cop_mes.values * tau_mult
alt['Línea base = predicción del modelo']          = ELEG.yhat0_modelo.values * tau_mult
alt['Línea base = media geométrica de ambas']      = np.sqrt(ELEG.ventas_femsa_cop_mes.values*ELEG.yhat0_modelo.values)*tau_mult
filas_s = []
for nm, va in alt.items():
    g = MARGEN*va - COSTO_NEVERA
    top = np.argsort(-g)[:N_NEVERAS]
    ids = set(ELEG.id_tienda.values[top])
    filas_s.append({'supuesto': nm, 'ganancia_total': g[top].sum(),
                    'coincide_con_lista': len(ids & set(LISTA.id_tienda))})
# (b) margenes alternativos
for m_alt in [0.20, 0.30]:
    g = m_alt*ELEG.ventas_adicionales_cop.values - COSTO_NEVERA
    top = np.argsort(-g)[:N_NEVERAS]
    filas_s.append({'supuesto': f'Margen {m_alt:.0%} en vez de 25%', 'ganancia_total': g[top].sum(),
                    'coincide_con_lista': len(set(ELEG.id_tienda.values[top]) & set(LISTA.id_tienda))})
# (c) costo alternativo
for c_alt in [40_000, 90_000]:
    g = MARGEN*ELEG.ventas_adicionales_cop.values - c_alt
    top = np.argsort(-g)[:N_NEVERAS]
    filas_s.append({'supuesto': f'Costo nevera {cop(c_alt)}', 'ganancia_total': g[top].sum(),
                    'coincide_con_lista': len(set(ELEG.id_tienda.values[top]) & set(LISTA.id_tienda))})
# (d) efecto 25% menor (escenario conservador)
g = MARGEN*ELEG.ventas_adicionales_cop.values*0.75 - COSTO_NEVERA
top = np.argsort(-g)[:N_NEVERAS]
filas_s.append({'supuesto': 'Efecto 25 % menor al estimado', 'ganancia_total': g[top].sum(),
                'coincide_con_lista': len(set(ELEG.id_tienda.values[top]) & set(LISTA.id_tienda))})

sens_dec = pd.DataFrame(filas_s)
print(sens_dec.to_string(index=False, formatters={'ganancia_total':'{:,.0f}'.format}))
print('\nLa composición de la lista es muy estable: los supuestos económicos (margen y costo) cambian')
print('la ganancia pero casi no cambian QUÉ tiendas se eligen, porque reescalan a todas por igual.')

# (e) que pasaria si priorizaramos tambien la retencion
valor_ret = EFECTIVIDAD_RETENCION * ELEG.prob_cambio.values * VALOR_EVITAR_CAMBIO - COSTO_RETENCION
ELEG['ganancia_con_retencion'] = ELEG.ganancia_estimada_cop_mes + np.maximum(valor_ret, 0)
lista_ret = ELEG.nlargest(N_NEVERAS, 'ganancia_con_retencion')
print(f'\nEscenario alternativo: si FEMSA valorara también la retención en el mismo ranking,')
print(f'  coincidirían {len(set(lista_ret.id_tienda)&set(LISTA.id_tienda)):,} de las 1.000 tiendas')
print(f'  y {(lista_ret.riesgo_cambio=="Alto").sum():,} serían de riesgo Alto '
      f'(vs {(LISTA.riesgo_cambio=="Alto").sum():,} en nuestra lista).')
print('  No lo usamos para la entrega porque la fórmula de ganancia del caso no incluye la retención,')
print('  pero queda como recomendación operativa: cruzar la lista con la campaña de retención.')
sens_dec.to_csv(f'{RUTA_SALIDAS}/sensibilidad_decision.csv', index=False)

### Lectura del módulo 6

**La recomendación.** Instalar las 1.000 neveras en la lista del modelo causal. La ganancia mensual estimada
es de **COP 251,9 millones** (COP 3.023 millones al año), con un retorno de **4,2x** sobre el costo de las
neveras. Supera a la regla de mayores ventas en **+20,1 %** (COP 209,8 millones) y a la de mayor tráfico en
**+54,7 %** (COP 162,8 millones). La diferencia proviene de un cambio de criterio concreto: **dejar de
comprar tamaño y empezar a comprar capacidad de respuesta**.

**Dónde está exactamente la diferencia.** Las dos listas comparten 638 tiendas; las 362 que las separan
cuentan la historia completa:

| | Solo en nuestra lista | Solo en la regla de ventas |
|---|---|---|
| Temperatura media | **26,6 °C** | 19,5 °C |
| Ventas actuales | COP 1.501.867 | **COP 2.201.314** |
| Ganancia estimada | **COP 175.892** | COP 59.487 |

La regla de ventas escoge tiendas con **COP 700.000 más** de venta mensual y obtiene **tres veces menos
ganancia**. Está pagando por tamaño en ciudades frías, donde una nevera añade poco. Además incluye 36
tiendas con ganancia negativa (la de tráfico, 95); nuestra lista, ninguna.

**Por qué las reglas simples pierden.** Las dos escogen tiendas grandes. El problema es que el tamaño ya
está correlacionado con tener nevera (por eso había sesgo de selección), y entre las *elegibles* las más
grandes son con frecuencia tiendas que ya tienen nevera propia o que están en ciudades frías. Nuestra lista
sacrifica ventas actuales a cambio de un efecto mucho mayor.

**El límite de 1.000 neveras es la restricción activa.** Hay **4.493** tiendas elegibles rentables y solo
1.000 neveras: alcanzan para el 22 % de ellas. No sobra capacidad, así que cada nevera asignada a la tienda
equivocada tiene un costo de oportunidad real. Esto también significa que **ampliar el plan sería
rentable**: la tienda número 1.000 todavía deja COP 119.003 al mes, muy por encima de cero.

**La decisión es robusta.** Cambiar el margen, el costo de la nevera o reducir el efecto estimado en 25 %
mueve la ganancia total pero casi no cambia *qué* tiendas se eligen, porque esos supuestos reescalan a todas
las tiendas por igual. El riesgo real no está en los supuestos económicos sino en el supuesto de
ignorabilidad del módulo 4.

---
# Módulo 7 — Exportación de resultados

Esta sección produce los archivos de la entrega y los insumos del tablero. Todo queda en la carpeta
`salidas/`.

| Archivo | Para qué | Filas |
|---|---|---|
| `lista_grupoX.csv` | **Entregable oficial**: las 1.000 tiendas | 1.000 |
| `tablero_tiendas.csv` | Hoja principal del tablero: todas las elegibles con su efecto, ganancia y riesgo | 12.700 |
| `tablero_kpis.csv` | Indicadores de resumen para las tarjetas del tablero | ~20 |
| `comparacion_reglas.csv` | Nuestra lista vs las reglas simples | 4 |
| `importancia_variables.csv` | Importancia en el modelo de ventas | ~47 |
| `importancia_heterogeneidad.csv` | Qué determina el tamaño del efecto | ~41 |
| `importancia_riesgo.csv` | Predictores del cambio de proveedor | ~107 |
| `metricas_modelos_ventas.csv` | Comparación de algoritmos del módulo 3 | 4 |
| `metricas_modelos_riesgo.csv` | Comparación de algoritmos del módulo 5 | 3 |
| `validacion_gate.csv` | Validación del modelo causal por decil | 10 |
| `curva_umbral_retencion.csv` | Valor neto de la retención por umbral | ~66 |
| `balance_covariables.csv` | SMD antes/después del ajuste | ~41 |
| `sensibilidad_efecto.csv`, `sensibilidad_decision.csv` | Robustez | ~5 / ~9 |
| `diccionario_salidas.csv` | Qué significa cada columna del tablero | ~25 |

In [ ]:
# ----------------------------- 7.1 Entregable oficial: lista_grupoX.csv -----------------------------
# Exactamente 1.000 filas y las tres columnas que pide el enunciado.
lista_entrega = (LISTA[['id_tienda','ganancia_estimada_cop_mes','riesgo_cambio']]
                 .copy()
                 .sort_values('ganancia_estimada_cop_mes', ascending=False)
                 .reset_index(drop=True))
lista_entrega['ganancia_estimada_cop_mes'] = lista_entrega.ganancia_estimada_cop_mes.round(0).astype('int64')

# --- VALIDACIONES DEL ENTREGABLE (una lista mal armada se califica con 0) ---
ids_elegibles = set(df.loc[df.elegible, 'id_tienda'])
v = {
 'exactamente 1.000 filas'          : len(lista_entrega) == 1000,
 'sin id_tienda repetidos'          : lista_entrega.id_tienda.duplicated().sum() == 0,
 'todas las tiendas son elegibles'  : set(lista_entrega.id_tienda).issubset(ids_elegibles),
 'ninguna tiene nevera FEMSA'       : df.set_index('id_tienda').loc[lista_entrega.id_tienda,'nevera_femsa'].sum() == 0,
 'regla Postobón y < 12 m² respetada':
     not ((df.set_index('id_tienda').loc[lista_entrega.id_tienda,'nevera_postobon'] == 1) &
          (df.set_index('id_tienda').loc[lista_entrega.id_tienda,'area_m2'] < AREA_MIN_SI_POSTOBON)).any(),
 'columnas exactas'                 : list(lista_entrega.columns) == ['id_tienda','ganancia_estimada_cop_mes','riesgo_cambio'],
 'sin valores faltantes'            : int(lista_entrega.isna().sum().sum()) == 0,
 'riesgo_cambio solo Alto/Bajo'     : set(lista_entrega.riesgo_cambio.unique()) <= {'Alto','Bajo'},
}
print('===== VALIDACIÓN DEL ENTREGABLE =====\n')
for k, ok in v.items():
    print(('  OK  ' if ok else ' FALLA') + f' | {k}')
assert all(v.values()), 'El entregable no cumple los requisitos.'

ruta_lista = f'{RUTA_SALIDAS}/lista_grupo{GRUPO}.csv'
lista_entrega.to_csv(ruta_lista, index=False, encoding='utf-8')
print(f'\nGuardado: {ruta_lista}')
print(f'  {len(lista_entrega):,} filas  |  ganancia total {cop(lista_entrega.ganancia_estimada_cop_mes.sum())} / mes')
print(f'  riesgo Alto: {(lista_entrega.riesgo_cambio=="Alto").sum()}  |  '
      f'riesgo Bajo: {(lista_entrega.riesgo_cambio=="Bajo").sum()}')
print('\nPrimeras 10 filas:')
print(lista_entrega.head(10).to_string(index=False))
if GRUPO == 'X':
    print('\n  *** RECORDATORIO: cambiar la variable GRUPO en el módulo 0 por su número de grupo ***')

In [ ]:
# ----------------------------- 7.2 Hoja principal del tablero -----------------------------
# Un archivo con TODAS las elegibles (no solo las 1.000) para que el tablero permita explorar
# la decision: filtros por region/ciudad/tipo, mapa, consulta por id_tienda y comparacion de reglas.
COLS_TABLERO = ['id_tienda','region','departamento','ciudad','zona','tipo_negocio',
                'latitud','longitud','estrato','temperatura_media_c','area_m2','poblacion_500m',
                'clientes_prom_dia','n_referencias_total','antiguedad_anios','num_empleados',
                'nevera_postobon','nevera_propia_tienda','nevera_otras_marcas',
                'visitas_mes_femsa','pct_quiebre_femsa','indice_precio_femsa',
                'ventas_femsa_cop_mes','ventas_postobon_cop_mes','share_femsa_valor']

tablero = ELEG[[c for c in COLS_TABLERO if c in ELEG.columns]].copy()
tablero['efecto_pct_ventas']        = ((np.exp(ELEG.tau_log)-1)*100).round(2)
tablero['ventas_adicionales_cop_mes'] = ELEG.ventas_adicionales_cop.round(0).astype('int64')
tablero['ganancia_estimada_cop_mes']  = ELEG.ganancia_estimada_cop_mes.round(0).astype('int64')
tablero['ranking']                  = ELEG.ranking
tablero['seleccionada']             = np.where(ELEG.seleccionada, 'Sí', 'No')
tablero['prob_cambio']              = ELEG.prob_cambio.round(4)
tablero['riesgo_cambio']            = ELEG.riesgo_cambio
tablero['rentable']                 = np.where(ELEG.ganancia_estimada_cop_mes > 0, 'Sí', 'No')
# marcas de las reglas simples, para que el tablero compare sin recalcular
tablero['en_regla_ventas']          = np.where(tablero.id_tienda.isin(regla_ventas.id_tienda), 'Sí', 'No')
tablero['en_regla_trafico']         = np.where(tablero.id_tienda.isin(regla_trafico.id_tienda), 'Sí', 'No')
# redondeo de montos para que Google Sheets no los lea como texto
for c in ['ventas_femsa_cop_mes','ventas_postobon_cop_mes']:
    tablero[c] = tablero[c].round(0).astype('int64')

tablero = tablero.sort_values('ranking').reset_index(drop=True)
tablero.to_csv(f'{RUTA_SALIDAS}/tablero_tiendas.csv', index=False, encoding='utf-8')
print(f'Guardado: {RUTA_SALIDAS}/tablero_tiendas.csv')
print(f'  {len(tablero):,} filas x {tablero.shape[1]} columnas (todas las elegibles)')
print(f'  seleccionadas: {(tablero.seleccionada=="Sí").sum():,}')
print('\nColumnas:')
print(textwrap.fill(', '.join(tablero.columns), 100))
print('\nPrimeras 5 filas (columnas clave):')
print(tablero[['id_tienda','region','ciudad','tipo_negocio','ventas_femsa_cop_mes',
               'efecto_pct_ventas','ganancia_estimada_cop_mes','ranking','seleccionada',
               'riesgo_cambio']].head().to_string(index=False))

In [ ]:
# ----------------------------- 7.3 KPIs para las tarjetas del tablero -----------------------------
kpis = [
 ('Neveras a instalar',                     N_NEVERAS,                                 'tiendas'),
 ('Tiendas elegibles',                      len(ELEG),                                 'tiendas'),
 ('Tiendas elegibles rentables',            int((ELEG.ganancia_estimada_cop_mes>0).sum()), 'tiendas'),
 ('Ganancia mensual total estimada',        round(G_TOTAL),                            'COP/mes'),
 ('Ganancia anual estimada',                round(G_TOTAL*12),                         'COP/año'),
 ('Ganancia promedio por tienda',           round(LISTA.ganancia_estimada_cop_mes.mean()), 'COP/mes'),
 ('Ganancia de la tienda n° 1.000',         round(LISTA.ganancia_estimada_cop_mes.min()),  'COP/mes'),
 ('Ventas adicionales totales',             round(LISTA.ventas_adicionales_cop.sum()),  'COP/mes'),
 ('Costo total de las neveras',             N_NEVERAS*COSTO_NEVERA,                     'COP/mes'),
 ('Retorno sobre el costo',                 round(G_TOTAL/(N_NEVERAS*COSTO_NEVERA), 2), 'veces'),
 ('Efecto promedio de la nevera',           round((np.exp(ate_dr)-1)*100, 1),           '% de ventas'),
 ('Efecto por diferencia simple (sesgado)', round((np.exp(res['1. Diferencia simple de medias']['log'])-1)*100, 1), '% de ventas'),
 ('Ganancia regla "mayores ventas"',        round(g_v),                                 'COP/mes'),
 ('Ganancia regla "mayor tráfico"',         round(g_t),                                 'COP/mes'),
 ('Ganancia selección aleatoria',           round(g_a),                                 'COP/mes'),
 ('Ventaja sobre la mejor regla simple',    round(g_ml - max(g_v, g_t)),                'COP/mes'),
 ('Ventaja sobre la mejor regla simple (%)',round((g_ml/max(g_v,g_t)-1)*100, 1),        '%'),
 ('Tiendas de riesgo Alto en la lista',     int((LISTA.riesgo_cambio=='Alto').sum()),   'tiendas'),
 ('Umbral de riesgo por costos',            UMBRAL_COSTOS,                              'probabilidad'),
 ('R² del mejor modelo de ventas',          round(mejor_r2, 4),                         'R² en test (log)'),
 ('ROC-AUC del modelo de riesgo',           round(tab_ch.ROC_AUC.iloc[0], 4),           'AUC'),
 ('PR-AUC del modelo de riesgo',            round(tab_ch.PR_AUC.iloc[0], 4),            'AUC'),
]
kpi_df = pd.DataFrame(kpis, columns=['indicador','valor','unidad'])
kpi_df.to_csv(f'{RUTA_SALIDAS}/tablero_kpis.csv', index=False, encoding='utf-8')
print('===== INDICADORES PARA EL TABLERO =====\n')
print(kpi_df.assign(valor=kpi_df.valor.map(lambda x: f'{x:,.4f}'.rstrip('0').rstrip('.')
                                           if isinstance(x, float) else f'{x:,}'))
      .to_string(index=False, justify='left'))

# Agregados por region / ciudad / tipo, listos para graficar en el tablero
for dim in ['region','ciudad','tipo_negocio','zona']:
    agg = (tablero[tablero.seleccionada=='Sí'].groupby(dim)
           .agg(tiendas=('id_tienda','size'),
                ganancia_total=('ganancia_estimada_cop_mes','sum'),
                ganancia_promedio=('ganancia_estimada_cop_mes','mean'),
                efecto_pct_promedio=('efecto_pct_ventas','mean'),
                riesgo_alto=('riesgo_cambio', lambda s:(s=='Alto').sum()))
           .round(1).sort_values('ganancia_total', ascending=False).reset_index())
    agg.to_csv(f'{RUTA_SALIDAS}/tablero_por_{dim}.csv', index=False, encoding='utf-8')
print(f'\nGuardados: tablero_kpis.csv y agregados por region, ciudad, tipo_negocio y zona')

In [ ]:
# ----------------------------- 7.4 Diccionario de las salidas -----------------------------
dicc_salidas = pd.DataFrame([
 ('id_tienda','Identificador único de la tienda','texto'),
 ('region / departamento / ciudad','Ubicación comercial y administrativa','texto'),
 ('zona','Urbana, Periurbana o Rural','texto'),
 ('tipo_negocio','Formato del punto de venta','texto'),
 ('latitud / longitud','Coordenadas para el mapa del tablero','grados'),
 ('estrato','Estrato socioeconómico predominante (1 a 6)','entero'),
 ('temperatura_media_c','Temperatura media de la ubicación','°C'),
 ('area_m2','Área del local','m²'),
 ('poblacion_500m','Personas en un radio de 500 m','personas'),
 ('clientes_prom_dia','Clientes promedio por día abierto','clientes/día'),
 ('n_referencias_total','Referencias en el surtido','referencias'),
 ('nevera_postobon','La tienda tiene nevera de Postobón (0/1)','binaria'),
 ('nevera_propia_tienda','El tendero tiene nevera propia (0/1)','binaria'),
 ('ventas_femsa_cop_mes','Ventas actuales de FEMSA, sin nevera','COP/mes'),
 ('efecto_pct_ventas','Aumento estimado de ventas al instalar la nevera','%'),
 ('ventas_adicionales_cop_mes','Ventas adicionales estimadas por la nevera','COP/mes'),
 ('ganancia_estimada_cop_mes','0,25 × ventas adicionales − 60.000','COP/mes'),
 ('ranking','Posición de la tienda al ordenar por ganancia estimada (1 = mejor)','entero'),
 ('seleccionada','Sí = está entre las 1.000 elegidas','Sí/No'),
 ('rentable','Sí = la ganancia estimada es positiva','Sí/No'),
 ('prob_cambio','Probabilidad estimada de cambiar de proveedor en 6 meses','0 a 1'),
 ('riesgo_cambio','Alto si prob_cambio ≥ 0,25 (umbral por costos del caso)','Alto/Bajo'),
 ('en_regla_ventas','Sí = la regla simple de mayores ventas también la elegiría','Sí/No'),
 ('en_regla_trafico','Sí = la regla simple de mayor tráfico también la elegiría','Sí/No'),
], columns=['columna','definicion','unidad'])
dicc_salidas.to_csv(f'{RUTA_SALIDAS}/diccionario_salidas.csv', index=False, encoding='utf-8')
print('Guardado: diccionario_salidas.csv')
print(dicc_salidas.to_string(index=False))

In [ ]:
# ----------------------------- 7.5 Resumen final e inventario de archivos -----------------------------
print('='*78)
print('RESUMEN EJECUTIVO'.center(78))
print('='*78)
dif_simple_pct = (np.exp(res['1. Diferencia simple de medias']['log'])-1)
resumen_txt = [
 '',
 'PREGUNTA   En cuales 1.000 tiendas instalar las neveras y cuanto se gana al mes.',
 '',
 f'RESPUESTA  Las 1.000 tiendas de {RUTA_SALIDAS}/lista_grupo{GRUPO}.csv.',
 f'           Ganancia mensual estimada: {cop(G_TOTAL)}  ({mm(G_TOTAL)} COP)',
 f'           Ganancia anual estimada:   {mm(G_TOTAL*12)} COP',
 f'           Retorno sobre el costo de las neveras: {G_TOTAL/(N_NEVERAS*COSTO_NEVERA):.2f}x',
 '',
 f'METODO     El efecto de la nevera se estima con un DR-learner (doblemente robusto)',
 f'           con cross-fitting sobre {len(X_CAUSAL)} variables previas al tratamiento. El efecto',
 f'           promedio es {(np.exp(ate_dr)-1):+.1%} de las ventas, frente a {dif_simple_pct:+.1%} que sugiere la',
 f'           diferencia simple de promedios: el sesgo de seleccion explica cerca de',
 f'           la mitad de esa brecha.',
 '',
 f'VALOR      Frente a la mejor regla simple, el modelo agrega {mm(g_ml-max(g_v,g_t))} COP/mes',
 f'           ({g_ml/max(g_v,g_t)-1:+.1%}), equivalente a {mm((g_ml-max(g_v,g_t))*12)} COP al anio.',
 '',
 f'PERFIL     Tiendas calientes ({LISTA.temperatura_media_c.mean():.1f} C de media), de alto trafico',
 f'           ({LISTA.clientes_prom_dia.mean():.0f} clientes/dia) y sin espacio frio propio: solo',
 f'           {LISTA.nevera_propia_tienda.mean():.0%} tiene nevera propia.',
 '',
 f'RIESGO     {(LISTA.riesgo_cambio=="Alto").sum():,} de las 1.000 tiendas tienen riesgo alto de cambiar de',
 f'           proveedor en 6 meses. Se recomienda cruzarlas con la campania de retencion.',
 '',
]
print(chr(10).join(resumen_txt))
print('='*78)

arch = sorted(os.listdir(RUTA_SALIDAS))
print(f'\nARCHIVOS GENERADOS EN {RUTA_SALIDAS}/ ({len(arch)}):\n')
for a in arch:
    print(f'  {os.path.getsize(f"{RUTA_SALIDAS}/{a}")/1024:>8.1f} KB  {a}')

In [ ]:
# ----------------------------- 7.6 Descargar los archivos (solo en Colab) -----------------------------
# Descomentar el bloque que necesiten.

# --- A) Descargar solo el entregable oficial ---
# from google.colab import files
# files.download(f'{RUTA_SALIDAS}/lista_grupo{GRUPO}.csv')

# --- B) Descargar todas las salidas en un ZIP ---
# import shutil
# shutil.make_archive('salidas_proyecto_neveras', 'zip', RUTA_SALIDAS)
# from google.colab import files
# files.download('salidas_proyecto_neveras.zip')

# --- C) Copiar las salidas a Google Drive (requiere drive.mount en el módulo 0) ---
# import shutil
# destino = '/content/drive/MyDrive/proyecto_ml/salidas'
# os.makedirs(destino, exist_ok=True)
# for a in os.listdir(RUTA_SALIDAS):
#     shutil.copy(f'{RUTA_SALIDAS}/{a}', destino)
# print('Copiado a', destino)

print('Cuaderno terminado.')
print(f'\nEntregable oficial: {RUTA_SALIDAS}/lista_grupo{GRUPO}.csv')
print(f'Insumo del tablero: {RUTA_SALIDAS}/tablero_tiendas.csv')
print('\nPara el tablero en Google Sheets:')
print('  Archivo > Importar > subir tablero_tiendas.csv > Insertar nueva hoja')
print('  Repetir con tablero_kpis.csv y comparacion_reglas.csv.')

---
## Cómo sigue el trabajo del grupo

**Para armar el tablero** (Streamlit, Dash o Power BI / Google Sheets):

- `tablero_tiendas.csv` es la hoja principal: trae las **12.700 elegibles** con su efecto estimado,
  ganancia, ranking, riesgo y coordenadas. La columna `seleccionada` marca las 1.000 elegidas.
  Alcanza para los requisitos del enunciado:
  - *filtros por región, ciudad y tipo de negocio* → columnas `region`, `ciudad`, `tipo_negocio`
  - *mapa de las 1.000 tiendas* → `latitud`, `longitud`, filtrando `seleccionada = Sí`
  - *ganancia estimada* → `ganancia_estimada_cop_mes`
  - *riesgo de cambio* → `riesgo_cambio`, `prob_cambio`
  - *consulta por id_tienda* → `id_tienda`
- `comparacion_reglas.csv` → la sección *comparación con reglas simples*.
- `importancia_variables.csv` e `importancia_heterogeneidad.csv` → la sección *importancia de variables*.
  La segunda es la más interesante para el tablero: muestra qué hace que una nevera rinda.
- `tablero_kpis.csv` → tarjetas de resumen.
- `tablero_por_region.csv`, `tablero_por_ciudad.csv`, `tablero_por_tipo_negocio.csv`,
  `tablero_por_zona.csv` → gráficos agregados sin recalcular nada.

**Para el informe gerencial:** el resumen ejecutivo de la celda 7.5 trae las cifras principales ya
redactadas; los supuestos están en la portada y las limitaciones al final del módulo 4.

**Antes de entregar:** cambiar `GRUPO = 'X'` en el módulo 0 por el número del grupo, para que el archivo
salga con el nombre correcto.